# Do Type Designers See Text Differently? — Analysis notebook

Analysis code for *Do Type Designers See Text Differently? Short-Exposure Text Recognition, an Expert vs. Novice Performance Study* (Mangas, Tashk, Bækgaard & Beier).

**Design:** 2 (Font: Helvetica vs. Klimax Plus; within-subjects) × 2 (Group: Novice vs. Expert; between-subjects). Whole report of five-letter random strings at eight nominal exposure durations (10–200 ms), analysed with a hierarchical Bayesian version of the Theory of Visual Attention (TVA) whole-report model.

**Data files** (paths are relative to the repository root; see `README.md`):

- `data/Data_Expert_Novice_Combined_Data.csv` — trial-level data, 36 participants × 160 trials. The typeface is stored under its PostScript name `KlimaxStd-Plus`; it is renamed to `Klimax Plus` immediately after loading (Section 2.1).
- `data/participant_info.csv` — participant demographics (age, gender, vision).

**Data-quality steps applied before modelling (Section 2):**

1. **Duplicate-response audit** — a letter typed twice but present only once in the stimulus is counted once (31 trials corrected).
2. **Participant exclusions** — R17_Novice (exposure-timing failure of about 100 ms) and R8_Expert (age matching); 34 participants remain. The exclusion of R8_Expert is revisited in a sensitivity refit (Section 9.4).
3. **Novice frame-snap, dominant-only mapping** — recorded novice exposure durations are snapped to the 60 Hz frame grid, and only trials whose snapped duration matches the dominant mapping for their intended duration are retained. Expert durations are used as recorded.

| Intended | Dominant corrected | Frames |
|----------|--------------------|--------|
| 10 ms    | 16.67 ms           | 1      |
| 20 ms    | 33.33 ms           | 2      |
| 30 ms    | 33.33 ms           | 2      |
| 40 ms    | 50.00 ms           | 3      |
| 50 ms    | 66.67 ms           | 4      |
| 60 ms    | 66.67 ms           | 4      |
| 90 ms    | 100.00 ms          | 6      |
| 200 ms   | 216.67 ms          | 13     |

**TVA parameters** (Bundesen, 1990):

- **K** — visual short-term memory storage capacity (letters)
- **C** — processing speed (letters/s)
- **t0** — perceptual threshold / minimum effective exposure (seconds in the model; reported in ms)

**Inferential hierarchy**, as stated in the manuscript: (1) posterior contrasts from the hierarchical model are primary; (2) frequentist tests on independently fitted per-participant parameters are secondary and Holm-adjusted; (3) dynamic time warping and the age-extremes comparison are descriptive only. No p-values are computed across posterior draws.

**Contents**

1. Setup
2. Data preparation
3. Hierarchical Bayesian TVA models
4. Primary analysis: posterior contrasts
5. H1 — Expert vs. Novice performance overall and per font
6. H3 — Age × expertise
7. H2 — Exposure-time differences between groups
8. Dynamic time warping and age × performance detail
9. Supplementary and sensitivity analyses
10. Where each manuscript figure and table is produced

Running the notebook from top to bottom reproduces every results figure (Figures 2–12) and table (Tables 2–5) of the manuscript. Figures are written to `figures/` (named by manuscript figure number) and tables to `results/tables/`. Posterior draws are cached as NetCDF files in `results/`; when those files are present the models are loaded instead of re-sampled.


## 1. Setup

In [ ]:
# 1. Setup: imports, repository paths and sampler configuration
# ============================================================
import os
import sys
import warnings
import importlib.metadata
from math import log
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az
import pytensor.tensor as pt

# Classical statistical analysis
from scipy import stats
from scipy.stats import pearsonr, spearmanr, mannwhitneyu, ttest_ind
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

from dtw import dtw as dtw_align

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ---- Repository paths --------------------------------------------------------
# Run the notebook from the repository root (the folder that contains `data/`).
# If it is launched from a sub-folder, the root is looked up one level above.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"           # cached posterior draws (NetCDF)
TABLE_DIR   = RESULTS_DIR / "tables"     # CSV versions of the reported tables
FIG_DIR     = ROOT / "figures"           # figures, named by manuscript figure number
for _d in (RESULTS_DIR, TABLE_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

TRIAL_DATA_FILE       = DATA_DIR / "Data_Expert_Novice_Combined_Data.csv"
PARTICIPANT_INFO_FILE = DATA_DIR / "participant_info.csv"
assert TRIAL_DATA_FILE.exists(), f"Trial data not found: {TRIAL_DATA_FILE}"
assert PARTICIPANT_INFO_FILE.exists(), f"Participant file not found: {PARTICIPANT_INFO_FILE}"


def savefig(name, dpi=150):
    """Save the current figure as figures/<name>.png."""
    plt.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight")


# ---- Reproducibility ---------------------------------------------------------
SEED = 1234
np.random.seed(SEED)   # jitter in strip plots and other np.random calls

# MCMC configuration. The reported results were obtained with the nutpie NUTS
# sampler (JAX backend). If nutpie/JAX are not installed, PyMC's own NUTS
# sampler is used with the same settings; posterior summaries then agree with
# the reported ones up to Monte-Carlo error rather than digit for digit.
SAMPLE_KWARGS = dict(
    chains=4, random_seed=SEED, target_accept=0.99,
    nuts={"max_treedepth": 14}, init="adapt_diag",
    return_inferencedata=True, progressbar=True,
)
try:
    import nutpie  # noqa: F401
    SAMPLE_KWARGS["nuts_sampler"] = "nutpie"
    try:
        import jax  # noqa: F401
        # PyMC >= 6 takes the computational backend as a top-level argument;
        # PyMC 5 (used for the reported results) takes it via nuts_sampler_kwargs.
        if int(pm.__version__.split(".")[0]) >= 6:
            SAMPLE_KWARGS["backend"] = "jax"
        else:
            SAMPLE_KWARGS["nuts_sampler_kwargs"] = {"backend": "jax"}
        SAMPLER_NAME = "nutpie (JAX backend)"
    except ImportError:
        SAMPLER_NAME = "nutpie (default backend)"
except ImportError:
    SAMPLER_NAME = "PyMC NUTS"

print(f"Repository root : {ROOT}")
print(f"Sampler         : {SAMPLER_NAME}")
print(f"Python          : {sys.version.split()[0]}")
print("Package versions:")
for _pkg in ["numpy", "pandas", "xarray", "scipy", "statsmodels", "pingouin",
             "matplotlib", "pymc", "pytensor", "arviz", "nutpie", "jax", "dtw-python"]:
    try:
        print(f"  {_pkg:<12s} {importlib.metadata.version(_pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {_pkg:<12s} (not installed)")


## 2. Data preparation

### 2.1 Trial-level data

In [ ]:
# 2.1 Trial-level data
# ============================================================
dat_raw = pd.read_csv(TRIAL_DATA_FILE)

# ---- Canonical typeface label ------------------------------------------------
# The CSV stores the PostScript name 'KlimaxStd-Plus'. The manuscript refers to
# the typeface as 'Klimax Plus', so the level is renamed ONCE, here, and
# 'Klimax Plus' is used everywhere downstream. The value is renamed in the data
# rather than only in plot labels because it is compared against in many
# places (dat['font'] == ..., dict keys, groupby levels). The font enters the
# model only through the numeric indicator isKlimax (0/1), and the rename
# changes neither row order nor the alphabetical order of the two levels.
FONT_RENAME = {"KlimaxStd-Plus": "Klimax Plus"}
dat_raw["Font"] = dat_raw["Font"].replace(FONT_RENAME)
print(f"Typeface labels after rename: {sorted(dat_raw['Font'].unique())}")

print(f"Raw data shape: {dat_raw.shape}")
print(f"Fonts: {dat_raw['Font'].unique()}")
print(f"Groups: {dat_raw['GroupCondition'].unique()}")
print(f"Participants: {sorted(dat_raw['ParticipantID'].unique())}")
print(f"Exposure times (ms): {sorted(dat_raw['TargetDisplay_ms'].unique())}")
display(dat_raw.head())

### 2.2 Participant demographics

In [ ]:
# 2.2 Participant demographics
# ============================================================
# data/participant_info.csv holds one row per recorded participant
# (Participants_ID = <ParticipantID>_<group>) with age, gender and vision.
participant_info_all = pd.read_csv(PARTICIPANT_INFO_FILE)
participant_info_all['vision'] = participant_info_all['vision'].str.capitalize()  # 'normal' -> 'Normal'

# Exclusions (see 2.5): R17_Novice (exposure-timing failure) and R8_Expert (age matching)
EXCLUDED_PARTICIPANTS = ['R17_Novice', 'R8_Expert']
participant_info = participant_info_all[
    ~participant_info_all['Participants_ID'].isin(EXCLUDED_PARTICIPANTS)
].reset_index(drop=True)
participant_info['age'] = participant_info['age'].astype(int)

print("=== Participant Demographics (after exclusions, N=34) ===")
print(participant_info.groupby('group')['age'].describe().round(1))
print(f"\nAge range — Novice: {participant_info[participant_info['group']=='Novice']['age'].min()}–"
      f"{participant_info[participant_info['group']=='Novice']['age'].max()}")
print(f"Age range — Expert: {participant_info[participant_info['group']=='Expert']['age'].min()}–"
      f"{participant_info[participant_info['group']=='Expert']['age'].max()}")
display(participant_info)


### 2.3 Duplicate-response audit

In [ ]:
# 2.3 Duplicate-response audit and correction
# ============================================================
# Check if any participant typed a letter multiple times and the scoring
# system counted each duplicate as a separate correct answer.
# E.g., Letters="icyjo", Response="juj" → 'j' typed twice, but 'j' only
# appears once in the stimulus. Original scoring: NumberCorrect=2 (jj).
# Corrected scoring: NumberCorrect=1 (j matched once, second j ignored).

def recount_correct(letters, responses):
    """Recount correct responses: each stimulus letter can only be matched once."""
    if responses in ('[]', '') or pd.isna(responses):
        return 0, ''
    letter_pool = list(str(letters))
    correct_chars = []
    for ch in str(responses):
        if ch in letter_pool:
            correct_chars.append(ch)
            letter_pool.remove(ch)  # consume — no double counting
    return len(correct_chars), ''.join(correct_chars)

# Apply to all trials
recounted = dat_raw.apply(
    lambda row: recount_correct(row['Letters'], row['Responses']), axis=1
)
dat_raw['NumberCorrect_new'] = recounted.apply(lambda x: x[0])
dat_raw['ItemsCorrect_new'] = recounted.apply(lambda x: x[1])
dat_raw['score_inflated'] = dat_raw['NumberCorrect'] - dat_raw['NumberCorrect_new']

# Report
n_inflated = (dat_raw['score_inflated'] > 0).sum()
total_inflation = dat_raw['score_inflated'].sum()

print("=" * 70)
print("DUPLICATE RESPONSE AUDIT")
print("=" * 70)
print(f"\nTrials with inflated NumberCorrect: {n_inflated} / {len(dat_raw)}")
print(f"Total extra 'correct' counts removed: {int(total_inflation)}")

if n_inflated > 0:
    inflated = dat_raw[dat_raw['score_inflated'] > 0]
    print(f"\n{'Letters':>10} {'Response':>10} {'Old':>4} {'New':>4} {'Participant':>20} {'Font':>16} {'t_ms':>6}")
    for _, r in inflated.iterrows():
        print(f"{r['Letters']:>10} {r['Responses']:>10} {r['NumberCorrect']:>4} "
              f"{r['NumberCorrect_new']:>4} {r['Participants_ID']:>20} {r['Font']:>16} {r['TargetDisplay_ms']:>6}")

    print(f"\nAffected trials by group:")
    print(inflated.groupby('GroupCondition').size())

# Apply correction: replace NumberCorrect with deduplicated count
dat_raw['NumberCorrect_original'] = dat_raw['NumberCorrect']
dat_raw['NumberCorrect'] = dat_raw['NumberCorrect_new']

print(f"\n→ NumberCorrect column updated with deduplicated scores.")
print(f"  Original scores saved in 'NumberCorrect_original'.")

### 2.4 Exposure-timing discrepancy (novice group)

In [ ]:
# 2.4 Exposure-timing discrepancy (novice group only)
# ============================================================
FRAME_MS = 1000.0 / 60.0

dat_raw['delta_t'] = dat_raw['TargetDisplay_chrono'] - dat_raw['TargetDisplay_ms']

print("=== Timing discrepancy by group ===")
for grp in ['Novice', 'Expert']:
    sub = dat_raw[dat_raw['GroupCondition'] == grp]
    dt = sub['delta_t']
    exceed = (abs(dt) > FRAME_MS).sum()
    print(f"\n  {grp} ({len(sub)} trials):")
    print(f"    mean Δt = {dt.mean():.1f} ms,  std = {dt.std():.1f} ms")
    print(f"    min = {dt.min()}, max = {dt.max()}")
    print(f"    |Δt| > 1 frame: {exceed} ({100*exceed/len(sub):.1f}%)")

# Novice frame-snap preview — including 83.33 and 200 ms breakdown
nov = dat_raw[dat_raw['GroupCondition'] == 'Novice'].copy()
nov = nov[~((nov['ParticipantID']=='R17'))].copy()
nov['n_frames'] = np.round(nov['TargetDisplay_chrono'] * 60.0 / 1000.0)
nov['t_snap'] = nov['n_frames'] * FRAME_MS
nov['snap_dev'] = np.abs(nov['t_snap'] - nov['TargetDisplay_ms'])

print(f"\n=== Novice: corrected time breakdown ===")
for tc in sorted(nov['t_snap'].unique()):
    s = nov[nov['t_snap'] == tc]
    keep = (s['snap_dev'] <= FRAME_MS + 0.01).sum()
    disc = (s['snap_dev'] > FRAME_MS + 0.01).sum()
    from_intended = s.groupby('TargetDisplay_ms').size().to_dict()
    marker = ' ← WILL EXCLUDE' if round(tc, 2) in [round(5*FRAME_MS, 2), 200.0] else ''
    print(f"  t_corr={tc:.2f}ms  total={len(s)}  keep={keep}  disc={disc}{marker}")
    for t_int, n in sorted(from_intended.items()):
        print(f"    from intended {t_int}ms → {n} trials")

print(f"\n  Note: t_corr ≈ 83.33ms (5 frames) and t_corr = 200.00ms (12 frames)")
print(f"  are minority bins for their intended conditions and will be excluded.")

### 2.5 Analysis dataset: participant exclusions and novice frame-snap (dominant-only mapping)

In [ ]:
# 2.5 Analysis dataset: participant exclusions and novice dominant-only frame-snap
# ============================================================
# Novice: frame-snap chrono, then KEEP ONLY trials whose corrected time
#         matches the DOMINANT mapping for their intended exposure time.
#         This excludes all cross-mapped minority trials, not just 83.33/200 ms.
# Expert: use original TargetDisplay_ms unchanged (timing was accurate).
#
# Dominant mapping (Novice):
#   Intended 10ms  → 16.67ms (1 frame)    |  Intended 50ms  → 66.67ms (4 frames)
#   Intended 20ms  → 33.33ms (2 frames)   |  Intended 60ms  → 66.67ms (4 frames)
#   Intended 30ms  → 33.33ms (2 frames)   |  Intended 90ms  → 100.00ms (6 frames)
#   Intended 40ms  → 50.00ms (3 frames)   |  Intended 200ms → 216.67ms (13 frames)
#
# Participant structure: 18 Novice + 18 Expert = 36 total.
# Excluded participants: R17 (Novice) and R8 (Expert) → 34 remain.

FRAME_MS = 1000.0 / 60.0

# Dominant corrected time for each intended Novice exposure
DOMINANT_MAP = {
    10:  round(1  * FRAME_MS, 4),   # 16.6667 ms
    20:  round(2  * FRAME_MS, 4),   # 33.3333 ms
    30:  round(2  * FRAME_MS, 4),   # 33.3333 ms
    40:  round(3  * FRAME_MS, 4),   # 50.0000 ms
    50:  round(4  * FRAME_MS, 4),   # 66.6667 ms
    60:  round(4  * FRAME_MS, 4),   # 66.6667 ms
    90:  round(6  * FRAME_MS, 4),   # 100.0000 ms
    200: round(13 * FRAME_MS, 4),   # 216.6667 ms
}

def build_analysis_data(raw: pd.DataFrame) -> pd.DataFrame:
    dat = raw.copy()

    # ----- Exclude participants -----
    exclude_ID    = ["R17", "R8"]
    exclude_Group = ["Novice", "Expert"]
    conditions = " or ".join(
        [f"(ParticipantID == '{pid}' and GroupCondition == '{grp}')"
         for pid, grp in zip(exclude_ID, exclude_Group)]
    )
    dat = dat.query(f"not ({conditions})").reset_index(drop=True)
    print(f"Excluded participants: {list(zip(exclude_ID, exclude_Group))}")

    # ----- Document group structure -----
    participants = [f"R{i}" for i in range(1, 19)]
    novice_ids = [pid for pid in participants if pid != exclude_ID[0]]
    expert_ids = [pid for pid in participants if pid != exclude_ID[1]]
    all_pairs = list(zip(["Novice"]*len(novice_ids), novice_ids)) + \
                list(zip(["Expert"]*len(expert_ids), expert_ids))
    print(f"\nAll (Group, ParticipantID) pairs ({len(all_pairs)} participants):")
    print(all_pairs)

    # ----- GROUP-SPECIFIC TIME CORRECTION -----
    nov_mask = dat['GroupCondition'] == 'Novice'
    exp_mask = dat['GroupCondition'] == 'Expert'
    n_novice_before = nov_mask.sum()

    # Novice: frame-snap correction
    dat.loc[nov_mask, 'n_frames'] = np.round(
        dat.loc[nov_mask, 'TargetDisplay_chrono'] * 60.0 / 1000.0
    )
    dat.loc[nov_mask, 't_corrected_ms'] = dat.loc[nov_mask, 'n_frames'] * FRAME_MS

    # Expert: no correction
    dat.loc[exp_mask, 't_corrected_ms'] = dat.loc[exp_mask, 'TargetDisplay_ms'].astype(float)

    # ----- Novice: keep ONLY dominant-mapped trials -----
    # For each Novice trial, check if its corrected time matches the
    # dominant mapping for its intended exposure time.
    novice_keep = pd.Series(False, index=dat.index)
    for t_int, t_dom in DOMINANT_MAP.items():
        match = (
            nov_mask &
            (dat['TargetDisplay_ms'] == t_int) &
            (np.abs(dat['t_corrected_ms'] - t_dom) < 0.1)
        )
        novice_keep |= match

    # Expert: keep all
    keep_mask = novice_keep | exp_mask
    n_novice_discarded = nov_mask.sum() - novice_keep.sum()

    # Report what was excluded
    print(f"\n--- Novice dominant-only filtering ---")
    print(f"{'Intended':>10} {'Dominant':>10} {'Kept':>6} {'Excluded':>8}")
    for t_int in sorted(DOMINANT_MAP.keys()):
        t_dom = DOMINANT_MAP[t_int]
        n_total = ((nov_mask) & (dat['TargetDisplay_ms'] == t_int)).sum()
        n_kept  = ((novice_keep) & (dat['TargetDisplay_ms'] == t_int)).sum()
        print(f"{t_int:>8d}ms {t_dom:>8.2f}ms {n_kept:>6d} {n_total - n_kept:>8d}")

    print(f"\nTotal Novice removed: {n_novice_discarded} / {n_novice_before} "
          f"({100*n_novice_discarded/n_novice_before:.1f}%)")
    print(f"Expert trials: all {exp_mask.sum()} kept")

    dat = dat[keep_mask].reset_index(drop=True)
    print(f"Total retained: {len(dat)}")

    # ----- Map columns to standard TVA names -----
    dat["t"] = dat["t_corrected_ms"]
    dat["t_intended"] = dat["TargetDisplay_ms"]
    dat["P"] = dat["NumberCorrect"]
    dat["S"] = 5
    dat["participant"] = dat["Participants_ID"]

    dat["font"]  = dat["Font"]
    dat["group"] = dat["GroupCondition"]

    dat["isKlimax"]       = (dat["font"] == "Klimax Plus").astype(int)
    dat["isExpert"]       = (dat["group"] == "Expert").astype(int)
    dat["isKlimaxExpert"] = dat["isKlimax"] * dat["isExpert"]

    dat["t_s"] = dat["t"] / 1000.0

    return dat

dat = build_analysis_data(dat_raw)

print(f"\nAnalysis data shape: {dat.shape}")
print(f"\n=== Final exposure times by group ===")
for grp in ['Novice', 'Expert']:
    sub = dat[dat['group']==grp]
    times = sorted(sub['t'].unique())
    counts = sub['t'].value_counts().sort_index()
    print(f"  {grp} ({len(sub)} trials):")
    for t in times:
        print(f"    {t:>8.2f} ms: {counts.get(t, 0):>4d} trials")

print(f"\nDummy coding check:")
print(dat[['font', 'group', 'isKlimax', 'isExpert', 'isKlimaxExpert']].drop_duplicates().sort_values(['font','group']))
display(dat.head())

# Merge age into dat
dat = dat.merge(
    participant_info[['Participants_ID', 'age', 'gender', 'vision']],
    left_on='participant', right_on='Participants_ID', how='left'
)
print(f"\nAge data merged: {dat['age'].notna().sum()} trials have age info")


### 2.6 Timing confusion matrix and retained trials (Figure S1)

In [ ]:
# 2.6 Confusion matrix of intended vs corrected novice times (Figure S1)
# ============================================================
import matplotlib.colors as mcolors

# --- A) Build confusion matrix from ALL Novice trials (before filtering) ---
# Re-derive from dat_raw to show what was kept vs excluded
FRAME_MS = 1000.0 / 60.0

nov_raw = dat_raw[
    (dat_raw['GroupCondition'] == 'Novice') &
    ~((dat_raw['ParticipantID'] == 'R17') & (dat_raw['GroupCondition'] == 'Novice'))
].copy()
nov_raw['n_frames'] = np.round(nov_raw['TargetDisplay_chrono'] * 60.0 / 1000.0)
nov_raw['t_corr'] = (nov_raw['n_frames'] * FRAME_MS).round(2)

# Full confusion matrix
confusion = pd.crosstab(
    nov_raw['TargetDisplay_ms'],
    nov_raw['t_corr'],
    margins=True
)
confusion.index.name = 'Intended (ms)'
confusion.columns.name = 'Corrected (ms)'

print("=== NOVICE: Full Confusion Matrix (Intended → Corrected) ===")
print(confusion.to_string())

# --- B) Heatmap visualization ---
conf_no_margins = confusion.iloc[:-1, :-1]  # drop 'All' row/col

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: full confusion matrix as heatmap
ax = axes[0]
data_arr = conf_no_margins.values.astype(float)
data_arr[data_arr == 0] = np.nan  # hide zeros
im = ax.imshow(data_arr, cmap='YlOrRd', aspect='auto',
               norm=mcolors.LogNorm(vmin=1, vmax=data_arr[~np.isnan(data_arr)].max()))
ax.set_xticks(range(len(conf_no_margins.columns)))
ax.set_xticklabels([f"{c:.1f}" for c in conf_no_margins.columns], rotation=45, ha='right')
ax.set_yticks(range(len(conf_no_margins.index)))
ax.set_yticklabels(conf_no_margins.index)
ax.set_xlabel('Corrected exposure (ms)')
ax.set_ylabel('Intended exposure (ms)')
ax.set_title('Novice: Intended → Corrected (all trials)')

# Annotate cells with counts and dominant/minority labels
DOMINANT_MAP = {10: round(1*FRAME_MS,2), 20: round(2*FRAME_MS,2),
                30: round(2*FRAME_MS,2), 40: round(3*FRAME_MS,2),
                50: round(4*FRAME_MS,2), 60: round(4*FRAME_MS,2),
                90: round(6*FRAME_MS,2), 200: round(13*FRAME_MS,2)}

for ri, t_int in enumerate(conf_no_margins.index):
    for ci, t_corr in enumerate(conf_no_margins.columns):
        val = conf_no_margins.iloc[ri, ci]
        if val > 0:
            is_dom = abs(t_corr - DOMINANT_MAP.get(t_int, -1)) < 0.1
            color = 'white' if val > 50 else 'black'
            marker = '✓' if is_dom else '✗'
            ax.text(ci, ri, f"{val}\n{marker}", ha='center', va='center',
                    fontsize=8, color=color, fontweight='bold' if is_dom else 'normal')
plt.colorbar(im, ax=ax, shrink=0.8, label='N trials')

# Right: After filtering — what remains
ax2 = axes[1]
nov_kept = dat[dat['group'] == 'Novice']
exp_kept = dat[dat['group'] == 'Expert']

print(f"\n=== After dominant-only filtering ===")
print(f"\nNovice retained times: {sorted(nov_kept['t'].unique())}")
print(f"Expert retained times: {sorted(exp_kept['t'].unique())}")

# Bar chart of trial counts per exposure time, by group
times_nov = nov_kept['t'].value_counts().sort_index()
times_exp = exp_kept['t'].value_counts().sort_index()
all_times = sorted(set(times_nov.index) | set(times_exp.index))

x = np.arange(len(all_times))
width = 0.35
bars_nov = [times_nov.get(t, 0) for t in all_times]
bars_exp = [times_exp.get(t, 0) for t in all_times]

ax2.bar(x - width/2, bars_nov, width, label='Novice', color='#1f77b4', alpha=0.8)
ax2.bar(x + width/2, bars_exp, width, label='Expert', color='#d62728', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([f"{t:.1f}" for t in all_times], rotation=45, ha='right')
ax2.set_xlabel('Exposure time (ms)')
ax2.set_ylabel('N trials')
ax2.set_title('Retained trials per exposure time (after filtering)')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

fig.suptitle('Novice Timing Correction: Confusion Matrix & Retained Data', fontsize=14, y=1.02)
fig.tight_layout()
savefig("FigS01_novice_timing_confusion_matrix")
plt.show()

### 2.7 Encodings, indices and prior centres

In [ ]:
# 2.7 Encodings, indices, helper functions and prior centres
# ============================================================
# With 18 Novice + 18 Expert (minus 2 excluded) = 34 unique participants,
# each identified by Participants_ID (e.g. R1_Novice, R3_Expert).

# Participant encoding
participants = dat['participant'].astype('category')
participant_idx = participants.cat.codes.values
n_participants = participants.cat.categories.size
participant_names = list(participants.cat.categories)

# Map each participant to their group (0=Novice, 1=Expert)
participant_group = (
    dat.groupby('participant')['isExpert'].first()
    .reindex(participant_names).values
)

n_novice  = int((participant_group == 0).sum())
n_expert  = int((participant_group == 1).sum())

print(f"Number of participants: {n_participants} ({n_novice} Novice + {n_expert} Expert)")
print(f"Participant names: {participant_names}")
print(f"Participant group (0=Novice, 1=Expert): {participant_group}")

# Observed data
y   = dat['P'].to_numpy()      # number correct
t_s = dat['t_s'].to_numpy()    # time in seconds
S   = dat['S'].to_numpy()      # set size per row

# Helper: smooth positive-part function
def pos(x):
    return 0.5 * (x + pt.sqrt(x**2))

def pos_stable(x, eps=1e-12):
    return 0.5 * (x + pm.math.sqrt(x * x + eps))

# Inverse logistic CDF
def qlogis(p):
    return np.log(p / (1 - p))

# Heuristic prior centers
mu_klin = qlogis(3/5)   # logit(0.6)
mu_clin = log(40)       # log(40)
mu_tlin = log(0.03)     # log(30 ms in seconds)

rng = np.random.default_rng(1234)

print(f"\nPrior centers: mu_klin={mu_klin:.3f}, mu_clin={mu_clin:.3f}, mu_tlin={mu_tlin:.3f}")
print(f"Total observations: {len(y)}")

### 2.8 Group indices for the hierarchical model

In [ ]:
# 2.8 Group indices for the hierarchical model
# ============================================================

font_idx = dat['isKlimax'].values
group_idx_per_participant = participant_group.astype(int)
n_groups = 2

print(f"n_participants: {n_participants}")
print(f"n_groups: {n_groups}")
print(f"Participants per group: {pd.Series(group_idx_per_participant).value_counts().sort_index().to_dict()}")
print(f"Group assignment: {group_idx_per_participant}")

## 3. Hierarchical Bayesian TVA models

### 3.1 Baseline model (participant random effects only)

No fixed effects for Font or Group — only participant-level variation around a grand mean.
This serves as the null model for the LOO-CV comparison against the full model.


In [ ]:
# 3.1 Baseline model (m_base)
# ============================================================
# The model is always defined (needed for compute_log_likelihood even when the
# cached posterior is loaded from results/).

filename_base = str(RESULTS_DIR / "posterior_baseline.nc")

with pm.Model() as m_base:
    # --- Priors on (global) intercepts on the latent scales ---
    klin_intercept = pm.Normal('klin_intercept', mu=mu_klin, sigma=1.5)
    clin_intercept = pm.Normal('clin_intercept', mu=mu_clin, sigma=1.0)
    tlin_intercept = pm.Normal('tlin_intercept', mu=mu_tlin, sigma=0.7)

    # --- Random intercept SDs ---
    sd_klin_part = pm.HalfNormal('sd_klin_part', sigma=0.7)
    sd_clin_part = pm.HalfNormal('sd_clin_part', sigma=0.7)
    sd_tlin_part = pm.HalfNormal('sd_tlin_part', sigma=0.7)

    # Non-centered parameterization for participant REs
    z_klin_part = pm.Normal('z_klin_part', 0, 1, shape=n_participants)
    z_clin_part = pm.Normal('z_clin_part', 0, 1, shape=n_participants)
    z_tlin_part = pm.Normal('z_tlin_part', 0, 1, shape=n_participants)

    re_klin = pm.Deterministic('re_klin', sd_klin_part * z_klin_part)
    re_clin = pm.Deterministic('re_clin', sd_clin_part * z_clin_part)
    re_tlin = pm.Deterministic('re_tlin', sd_tlin_part * z_tlin_part)

    # --- Linear predictors on latent scales ---
    klin_lin = klin_intercept + re_klin[participant_idx]
    clin_lin = clin_intercept + re_clin[participant_idx]
    tlin_lin = tlin_intercept + re_tlin[participant_idx]

    # --- Transformations to constrained parameters ---
    K  = S * pm.math.sigmoid(klin_lin)
    C  = pm.math.exp(clin_lin)
    t0 = pm.math.exp(tlin_lin)

    # ---- Stability tweaks ----
    K_safe = pm.math.clip(K, 1e-12, np.inf)
    dt     = t_s - t0
    pos_dt = pos_stable(dt, eps=1e-12)
    expo   = -(C / K_safe) * pos_dt
    expo   = pm.math.clip(expo, -1e6, 1e6)
    mu     = K * (1 - pm.math.exp(expo))

    sigma = pm.HalfStudentT('sigma', nu=3, sigma=2)
    nu    = pm.Gamma('nu', alpha=2, beta=0.1)

    P_obs = pm.StudentT('P_obs', nu=nu, mu=mu, sigma=sigma, observed=y)

# --- Sample or load cache ---
if os.path.exists(filename_base):
    print(f"Loading idata_base from {filename_base}...")
    idata_base = az.from_netcdf(filename_base)
else:
    print("Cache not found. Running PyMC sampling for m_base...")
    with m_base:
        idata_base = pm.sample(
            draws=3000, tune=1000,
            idata_kwargs={"log_likelihood": True},
            **SAMPLE_KWARGS,
        )
    idata_base.to_netcdf(filename_base)
    print(f"Baseline model sampling complete. Saved to {filename_base}.")

In [ ]:
# Log-likelihood for the baseline model (skipped if already present in the cache)
# ============================================================
if hasattr(idata_base, "log_likelihood"):
    print("log_likelihood group already present in idata_base — skipping.")
else:
    print("Computing log-likelihood for baseline model...")
    with m_base:
        idata_base = pm.compute_log_likelihood(
            idata_base, var_names=["P_obs"], extend_inferencedata=True
        )
    idata_base.to_netcdf(filename_base)
    print("Done. Re-saved to NetCDF with log_likelihood.")

### 3.2 Full model: Font × Group with participant random intercepts and random font slopes

Group-specific intercepts + population-level font effect + per-participant font random slopes + Font × Group interaction.

- **Baseline** = Helvetica, Novice
- **group_mean** = separate group-level means for Novice vs. Expert
- **font_effect** = population-level effect of Klimax Plus relative to Helvetica
- **fontgroup** = interaction: does expertise modulate the font legibility effect?
- **Random slopes** = per-participant deviations in font sensitivity


In [ ]:
# 3.2 Full model (m_full): Font x Group, participant random intercepts, random font slopes
# ============================================================

filename_full = str(RESULTS_DIR / "posterior_full.nc")

with pm.Model() as m_full:

    # ================================================================
    # LEVEL 1: Group means for each TVA parameter (on latent scales)
    # ================================================================
    group_mean_klin = pm.Normal('group_mean_klin', mu=mu_klin, sigma=1.5, shape=n_groups)
    group_mean_clin = pm.Normal('group_mean_clin', mu=mu_clin, sigma=1.0, shape=n_groups)
    group_mean_tlin = pm.Normal('group_mean_tlin', mu=mu_tlin, sigma=0.7, shape=n_groups)

    # ================================================================
    # LEVEL 2: Participant random intercepts (centered on their group)
    # ================================================================
    sd_klin_part = pm.HalfNormal('sd_klin_part', sigma=0.7)
    sd_clin_part = pm.HalfNormal('sd_clin_part', sigma=0.7)
    sd_tlin_part = pm.HalfNormal('sd_tlin_part', sigma=0.7)

    z_klin_part = pm.Normal('z_klin_part', 0, 1, shape=n_participants)
    z_clin_part = pm.Normal('z_clin_part', 0, 1, shape=n_participants)
    z_tlin_part = pm.Normal('z_tlin_part', 0, 1, shape=n_participants)

    part_klin = pm.Deterministic(
        'part_klin',
        group_mean_klin[group_idx_per_participant] + sd_klin_part * z_klin_part
    )
    part_clin = pm.Deterministic(
        'part_clin',
        group_mean_clin[group_idx_per_participant] + sd_clin_part * z_clin_part
    )
    part_tlin = pm.Deterministic(
        'part_tlin',
        group_mean_tlin[group_idx_per_participant] + sd_tlin_part * z_tlin_part
    )

    # ================================================================
    # FONT EFFECT: Population-level + random slopes per participant
    # ================================================================
    font_klin = pm.Normal('font_klin', mu=0, sigma=0.7)
    font_clin = pm.Normal('font_clin', mu=0, sigma=0.7)
    font_tlin = pm.Normal('font_tlin', mu=0, sigma=0.5)

    sd_font_klin = pm.HalfNormal('sd_font_klin', sigma=0.3)
    sd_font_clin = pm.HalfNormal('sd_font_clin', sigma=0.3)
    sd_font_tlin = pm.HalfNormal('sd_font_tlin', sigma=0.2)

    z_font_klin = pm.Normal('z_font_klin', 0, 1, shape=n_participants)
    z_font_clin = pm.Normal('z_font_clin', 0, 1, shape=n_participants)
    z_font_tlin = pm.Normal('z_font_tlin', 0, 1, shape=n_participants)

    # ================================================================
    # INTERACTION: Font × Group
    # ================================================================
    fontgroup_klin = pm.Normal('fontgroup_klin', mu=0, sigma=0.5)
    fontgroup_clin = pm.Normal('fontgroup_clin', mu=0, sigma=0.5)
    fontgroup_tlin = pm.Normal('fontgroup_tlin', mu=0, sigma=0.3)

    # ================================================================
    # LINEAR PREDICTOR
    # ================================================================
    isKlimax_obs = font_idx
    isExpert_obs = group_idx_per_participant[participant_idx]

    klin_lin = (
        part_klin[participant_idx]
        + (font_klin + sd_font_klin * z_font_klin[participant_idx]) * isKlimax_obs
        + fontgroup_klin * isKlimax_obs * isExpert_obs
    )
    clin_lin = (
        part_clin[participant_idx]
        + (font_clin + sd_font_clin * z_font_clin[participant_idx]) * isKlimax_obs
        + fontgroup_clin * isKlimax_obs * isExpert_obs
    )
    tlin_lin = (
        part_tlin[participant_idx]
        + (font_tlin + sd_font_tlin * z_font_tlin[participant_idx]) * isKlimax_obs
        + fontgroup_tlin * isKlimax_obs * isExpert_obs
    )

    # ================================================================
    # TVA EQUATION
    # ================================================================
    K  = S * pm.math.sigmoid(klin_lin)
    C  = pm.math.exp(clin_lin)
    t0 = pm.math.exp(tlin_lin)

    K_safe = pm.math.clip(K, 1e-12, np.inf)
    dt     = t_s - t0
    pos_dt = pos_stable(dt, eps=1e-12)
    expo   = -(C / K_safe) * pos_dt
    expo   = pm.math.clip(expo, -1e6, 1e6)
    mu     = K * (1 - pm.math.exp(expo))

    sigma = pm.HalfStudentT('sigma', nu=3, sigma=2)
    nu    = pm.Gamma('nu', alpha=2, beta=0.1)

    P_obs = pm.StudentT('P_obs', nu=nu, mu=mu, sigma=sigma, observed=y)

# --- Sample or load cache ---
if os.path.exists(filename_full):
    print(f"Loading idata_full from {filename_full}...")
    idata_full = az.from_netcdf(filename_full)
else:
    print("Cache not found. Running PyMC sampling for m_full...")
    with m_full:
        idata_full = pm.sample(
            draws=3000, tune=1500,
            idata_kwargs={"log_likelihood": True},
            **SAMPLE_KWARGS,
        )
    idata_full.to_netcdf(filename_full)
    print(f"m_full sampling complete. Saved to {filename_full}.")

In [ ]:
# Log-likelihood for the full model (skipped if already present in the cache)
# ============================================================
if hasattr(idata_full, "log_likelihood"):
    print("log_likelihood already present in idata_full — skipping.")
else:
    print("Computing log-likelihood for idata_full...")
    with m_full:
        pm.compute_log_likelihood(idata_full, progressbar=True)
    idata_full.to_netcdf(filename_full)
    print("Done. Saved updated idata_full.")

### 3.3 Model comparison (LOO-CV) and posterior predictive check

The baseline (intercept-only) model is compared against the full hierarchical model to test
whether the Font × Group structure explains meaningful variance beyond participant-level variation.


In [ ]:
# 3.3 Model comparison: baseline vs full model (LOO-CV); posterior predictive check
# ============================================================
# az.loo() returns an ELPDData object whose main attribute is 'elpd_loo'
# ('loo' in older ArviZ versions); the helper below handles both.

def loo_summary(idata, label):
    """Print LOO summary compatible with both old and new ArviZ API."""
    loo_result = az.loo(idata, pointwise=True)
    # attribute names differ between ArviZ generations (elpd_loo / p_loo in 0.x, elpd / p in 1.x)
    elpd = next((getattr(loo_result, a) for a in ('elpd_loo', 'loo', 'elpd')
                 if getattr(loo_result, a, None) is not None), None)
    se   = getattr(loo_result, 'se', None)
    p    = next((getattr(loo_result, a) for a in ('p_loo', 'p')
                 if getattr(loo_result, a, None) is not None), None)
    print(f"  {label}:")
    print(f"    ELPD LOO:  {elpd:.1f}" + (f"  (SE = {se:.1f})" if se is not None else ""))
    print(f"    p_loo:     {p:.1f}" if p is not None else "")
    k = loo_result.pareto_k.values
    print(f"    Pareto-k:  Good (≤0.7): {(k <= 0.7).sum()} ({(k <= 0.7).mean():.1%}), "
          f"Bad (0.7–1): {((k > 0.7) & (k <= 1.0)).sum()}, "
          f"Very bad (>1): {(k > 1.0).sum()}, "
          f"Max: {k.max():.3f}")
    return loo_result

print("=== Baseline LOO ===")
loo_base_pw = loo_summary(idata_base, "m_base")

print("\n=== Full model LOO ===")
loo_full_pw = loo_summary(idata_full, "m_full")

print("\n=== LOO Model Comparison (lower elpd_loo → worse; higher → better) ===")
try:
    comp = az.compare({'base': idata_base, 'full': idata_full}, ic='loo')
except TypeError:            # ArviZ >= 1 no longer takes `ic`
    comp = az.compare({'base': idata_base, 'full': idata_full})
print(comp)

# Posterior predictive check for the full model (progressbar=False keeps the
# saved notebook output compact)
with m_full:
    idata_full = pm.sample_posterior_predictive(
        idata_full, random_seed=1234, extend_inferencedata=True, progressbar=False
    )

ppc_axes = az.plot_ppc(idata_full, var_names=["P_obs"])
if hasattr(ppc_axes, "set_xlim"):
    ppc_axes.set_xlim((-10, 10))
plt.title("Posterior Predictive Check (Full model)")
plt.show()

### 3.4 Convergence diagnostics

In [ ]:
# 3.4 Diagnostics: effective sample size, posterior summary, trace and posterior plots
# ============================================================

print("=== Effective Sample Size ===")
display(az.ess(idata_full))

print("\n=== Posterior Summary ===")
display(az.summary(idata_full))

# Trace plots
az.plot_trace(idata_full)
plt.tight_layout()
plt.show()

# Posterior distributions
az.plot_posterior(idata_full)
plt.tight_layout()
plt.show()

# Forest plot
az.plot_forest(idata_full)
plt.tight_layout()
plt.show()

### 3.5 Pareto-k diagnostics (full model)

In [ ]:
# 3.5 Pareto-k diagnostics (full model)
# ============================================================

loo_full = az.loo(idata_full, pointwise=True)

# attribute names differ between ArviZ generations
elpd_val = next((getattr(loo_full, a) for a in ('elpd_loo', 'loo', 'elpd')
                 if getattr(loo_full, a, None) is not None), None)
p_loo_val = next((getattr(loo_full, a) for a in ('p_loo', 'p')
                  if getattr(loo_full, a, None) is not None), None)
print(f"ELPD LOO: {elpd_val:.1f}")
print(f"p_loo:    {p_loo_val:.1f}")

k_vals = loo_full.pareto_k.values
print(f"\nPareto-k summary:")
print(f"  Good  (≤0.7):  {(k_vals <= 0.7).sum()} ({(k_vals <= 0.7).mean():.1%})")
print(f"  Bad (0.7-1.0): {((k_vals > 0.7) & (k_vals <= 1.0)).sum()}")
print(f"  Very bad (>1): {(k_vals > 1.0).sum()}")
print(f"  Max Pareto-k:  {k_vals.max():.3f}")

# Cross-tabulate bad Pareto-k by exposure time and condition
dat_diag = dat.copy()
dat_diag['pareto_k'] = k_vals
bad = dat_diag[dat_diag['pareto_k'] > 0.7]
if len(bad) > 0:
    print(f"\n=== {len(bad)} bad Pareto-k observations ===")
    print("By exposure time:")
    print(bad.groupby('t').size())
    print("\nBy font × group:")
    print(bad.groupby(['font', 'group']).size())
    print("\nBy participant (top 10):")
    print(bad.groupby('participant').size().sort_values(ascending=False).head(10))
else:
    print("\nNo bad Pareto-k observations!")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(k_vals, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0.7, color='red', ls='--', label='k=0.7 threshold')
axes[0].set_xlabel('Pareto-k')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Pareto-k values')
axes[0].legend()

axes[1].scatter(range(len(k_vals)), k_vals, s=3, alpha=0.5)
axes[1].axhline(0.7, color='red', ls='--', label='k=0.7 threshold')
axes[1].set_xlabel('Observation index')
axes[1].set_ylabel('Pareto-k')
axes[1].set_title('Pareto-k by observation')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Primary analysis: posterior contrasts

### 4.1 Posterior extraction and TVA curve utilities

In [ ]:
# 4.1 Posterior extraction and TVA curve utilities
# ============================================================

def pos_part(x):
    return np.clip(x, 0.0, None)

sigmoid = lambda x: 1.0 / (1.0 + np.exp(-x))

t_grid = np.arange(0.0, 221.0, 1.0) / 1000.0  # 0..220 ms in seconds

def tva_curve(K, C, t0, t_grid):
    Kc = np.clip(K, 1e-12, None)
    dt = t_grid[None, :] - t0[:, None]
    expo = -(C[:, None] / Kc[:, None]) * pos_part(dt)
    expo = np.clip(expo, -1e6, 1e6)
    P = K[:, None] * (1.0 - np.exp(expo))
    return P

def median_and_ci(samples, alpha=0.05, axis=0):
    low  = np.quantile(samples, alpha / 2, axis=axis)
    med  = np.quantile(samples, 0.5, axis=axis)
    high = np.quantile(samples, 1 - alpha / 2, axis=axis)
    return med, low, high

fonts_list  = ["Helvetica", "Klimax Plus"]
groups_list = ["Novice", "Expert"]

# Extract full-model posteriors
post_full = idata_full.posterior

def _extract_1d(post, name):
    return post[name].stack(sample=("chain", "draw")).transpose("sample").values

def _extract_vec(post, name):
    da = post[name].stack(sample=("chain", "draw"))
    if da.dims[0] != "sample":
        da = da.transpose("sample", ...)
    return da.values

_gm_klin = _extract_vec(post_full, "group_mean_klin")
_gm_clin = _extract_vec(post_full, "group_mean_clin")
_gm_tlin = _extract_vec(post_full, "group_mean_tlin")

_font_klin = _extract_1d(post_full, "font_klin")
_font_clin = _extract_1d(post_full, "font_clin")
_font_tlin = _extract_1d(post_full, "font_tlin")

_fg_klin = _extract_1d(post_full, "fontgroup_klin")
_fg_clin = _extract_1d(post_full, "fontgroup_clin")
_fg_tlin = _extract_1d(post_full, "fontgroup_tlin")

def get_KCt0(font, group):
    g = 0 if group == "Novice" else 1
    isK = 1.0 if font == "Klimax Plus" else 0.0
    isE = 1.0 if group == "Expert" else 0.0

    klin = _gm_klin[:, g] + _font_klin * isK + _fg_klin * isK * isE
    clin = _gm_clin[:, g] + _font_clin * isK + _fg_clin * isK * isE
    tlin = _gm_tlin[:, g] + _font_tlin * isK + _fg_tlin * isK * isE

    return 5.0 * sigmoid(klin), np.exp(clin), np.exp(tlin)

print(f"Posterior samples: {_gm_klin.shape[0]}")
print("\n=== K posterior means (sanity check) ===")
for font in fonts_list:
    for group in groups_list:
        k = get_KCt0(font, group)[0].mean()
        print(f"  {font} + {group}: K = {k:.3f}")

### 4.2 Figure 2 — posterior TVA curves with empirical means

In [ ]:
# 4.2 Figure 2: posterior TVA curves + empirical means (group-specific exposure grids)
# ============================================================
# Novice empirical data: plotted at corrected (frame-snapped) times
# Expert empirical data: plotted at original intended times

dat_plot = dat.copy()
dat_plot["t_plot"] = dat_plot["t"] / 1000.0  # corrected for Novice, original for Expert

# Aggregate empirical means per (font, group, actual t used)
agg = (
    dat_plot
    .groupby(["font", "group", "t_plot"], as_index=False)
    .agg(mean_P=("P", "mean"), n=("P", "size"), std_P=("P", "std"))
)
agg["se_P"] = agg["std_P"] / np.sqrt(agg["n"].clip(lower=1))
z_val = 1.96
agg["lo"] = agg["mean_P"] - z_val * agg["se_P"]
agg["hi"] = agg["mean_P"] + z_val * agg["se_P"]

group_colors     = {"Novice": "#1f77b4", "Expert": "#d62728"}
group_linestyles = {"Novice": "-", "Expert": "--"}
group_markers    = {"Novice": "s", "Expert": "o"}

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

for col, font in enumerate(fonts_list):
    ax = axes[col]
    for group in groups_list:
        K, C, t0 = get_KCt0(font, group)
        P = tva_curve(K, C, t0, t_grid)
        P_med, P_lo, P_hi = median_and_ci(P, alpha=0.05, axis=0)

        ax.plot(t_grid * 1000, P_med, color=group_colors[group], lw=2,
                linestyle=group_linestyles[group], label=group)
        ax.fill_between(t_grid * 1000, P_lo, P_hi,
                        color=group_colors[group], alpha=0.15)

        sub = agg[(agg["font"] == font) & (agg["group"] == group)]
        ax.errorbar(sub["t_plot"] * 1000, sub["mean_P"],
                    yerr=[sub["mean_P"] - sub["lo"], sub["hi"] - sub["mean_P"]],
                    fmt=group_markers[group], color=group_colors[group],
                    markersize=6, capsize=3, alpha=0.7)
    ax.set_xticks([0, 10, 16.67, 20, 30, 33.33, 40, 50, 60, 66.67, 90, 100, 150, 200, 217])
    ax.tick_params(axis='x' , rotation=75)
    ax.set_title(f"{font}", fontsize=14, fontweight='bold')
    ax.set_ylim(-0.5, 5.5)
    ax.set_xlabel("Exposure duration (ms)\n(Novice: frame-snapped; Expert: original)")
    ax.grid(alpha=0.3)
    if col == 0:
        ax.set_ylabel("Number correct")
    if col == 1:
        ax.legend(loc='lower right', fontsize=10)

fig.suptitle("TVA Curves: Novice (corrected) vs Expert (original times)",
             fontsize=14, y=1.01)
fig.tight_layout()
savefig("Fig02_posterior_tva_curves")
plt.show()

### 4.3 Figure 9 — posterior difference curves Δ(t) on the common 0–200 ms grid (and Figure S2)

In [ ]:
# 4.3 Figure 9: posterior difference curves on the common 0-200 ms grid (and Figure S2)
# ============================================================
# Posterior draws are Monte-Carlo samples describing uncertainty about the
# parameters, not independent observations, so no significance tests are run
# across draws. The posterior difference curve
#     Delta(t) = P_Expert(t) - P_Novice(t)
# is summarised by its posterior mean, 95% credible interval, and the posterior
# probability of direction P(Delta(t) > 0). A time point is flagged as
# "credible" only when the 95% CrI excludes zero — a statement about the
# posterior, not a frequentist test.
#
# Novice curves are defined over 0-216.67 ms and Expert curves over 0-200 ms;
# the comparison is restricted to the shared 0-200 ms domain.

# Common time grid: 0..200 ms at 1 ms resolution
t_common = np.arange(0.0, 201.0, 1.0) / 1000.0  # 201 points in seconds
n_tc = len(t_common)
n_samples = _gm_klin.shape[0]

print(f"Common time grid: 0-{t_common[-1]*1000:.0f} ms ({n_tc} points)")
print(f"Posterior draws: {n_samples}  (used as uncertainty, NOT as observations)")

# Compute P(t) on common grid for all 4 conditions
curves_common = {}
for font in fonts_list:
    for group in groups_list:
        K, C, t0 = get_KCt0(font, group)
        P_all = tva_curve(K, C, t0, t_common)  # (n_samples, 201)
        curves_common[(font, group)] = P_all

# --- Posterior summaries of Delta(t) = Expert - Novice, per font ---
results_common = {}
for font in fonts_list:
    P_exp = curves_common[(font, "Expert")]
    P_nov = curves_common[(font, "Novice")]
    delta = P_exp - P_nov                       # (n_samples, n_tc)

    delta_mean = delta.mean(axis=0)
    delta_med  = np.median(delta, axis=0)
    delta_lo   = np.quantile(delta, 0.025, axis=0)
    delta_hi   = np.quantile(delta, 0.975, axis=0)
    p_dir      = (delta > 0).mean(axis=0)       # P(Delta(t) > 0)
    cred       = delta_lo > 0                   # 95% CrI excludes zero (above)

    results_common[font] = {
        'delta': delta, 'mean': delta_mean, 'median': delta_med,
        'lo': delta_lo, 'hi': delta_hi,
        'p_dir': p_dir, 'credible': cred,
    }

# --- FIGURE A: Delta(t) with 95% CrI, and probability of direction ---
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
t_ms_c = t_common * 1000

for col, font in enumerate(fonts_list):
    R = results_common[font]

    # Top row: posterior Delta(t) with credible interval
    ax = axes[0, col]
    ax.plot(t_ms_c, R['mean'], 'k-', lw=2, label=r'Posterior mean $\Delta(t)$ = Expert - Novice')
    ax.fill_between(t_ms_c, R['lo'], R['hi'], alpha=0.25, color='steelblue', label='95% CrI')
    ax.axhline(0, color='grey', ls='--', lw=1)

    # Shade only where the 95% CrI excludes zero
    cred = R['credible']
    for j in range(1, n_tc):
        if cred[j]:
            ax.axvspan(t_ms_c[j-1], t_ms_c[j], alpha=0.18, color='seagreen')
    n_cred = int(cred.sum())
    ax.set_title(f'{font}: Expert - Novice\n95% CrI excludes zero at {n_cred}/{n_tc} time points',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Exposure duration (ms) [common grid 0-200]')
    ax.set_ylabel(r'$\Delta$ Number correct')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)

    # Bottom row: posterior probability of direction (replaces -log10 p)
    ax2 = axes[1, col]
    ax2.plot(t_ms_c, R['p_dir'], 'k-', lw=1.5)
    ax2.axhline(0.5,  color='grey',   ls='--', lw=1,   label='P = 0.50 (no direction)')
    ax2.axhline(0.95, color='seagreen', ls='--', lw=1.5, label='P = 0.95')
    ax2.axhline(0.975, color='darkgreen', ls=':', lw=1.5,
                label='P = 0.975 (one-sided equiv. of 95% CrI)')
    ax2.set_ylim(0, 1.02)
    ax2.set_title(rf'{font}: $P(\Delta(t) > 0)$', fontsize=12)
    ax2.set_xlabel('Exposure duration (ms)')
    ax2.set_ylabel(r'$P(\Delta(t) > 0)$')
    ax2.legend(fontsize=8, loc='lower right')
    ax2.grid(alpha=0.3)

fig.suptitle('Posterior difference curves on the common grid (0-200 ms).\n'
             'No p-values: posterior draws quantify uncertainty, they are not observations.',
             fontsize=14, y=1.03)
fig.tight_layout()
savefig("Fig09_posterior_difference_curves")
plt.show()

# --- FIGURE B: Posterior Delta at a few representative exposure times ---
shared_ms = [10, 20, 30, 40, 50, 60, 90, 100, 150, 200]
shared_idx = [int(t) for t in shared_ms]

fig, axes = plt.subplots(len(fonts_list), len(shared_ms), figsize=(28, 8))
for row, font in enumerate(fonts_list):
    R = results_common[font]
    for ci, (t_idx, t_label) in enumerate(zip(shared_idx, shared_ms)):
        ax = axes[row, ci]
        d = R['delta'][:, t_idx]
        lo, hi = R['lo'][t_idx], R['hi'][t_idx]
        pdir = R['p_dir'][t_idx]
        color = '#2ca02c' if lo > 0 else '#1f77b4'
        ax.hist(d, bins=60, density=True, alpha=0.7, color=color, edgecolor='none')
        ax.axvline(0, color='black', ls='--', lw=1)
        ax.axvline(d.mean(), color='darkred', ls='-', lw=1.5)
        ax.set_title(f'{t_label} ms\nP(>0)={pdir:.2f}', fontsize=8)
        if ci == 0:
            ax.set_ylabel(font[:8], fontsize=9, fontweight='bold')
fig.suptitle(r'Posterior $\Delta(t)$ at selected exposure times '
             '(green = 95% CrI excludes zero)', fontsize=14, y=1.02)
fig.tight_layout()
savefig("FigS02_posterior_difference_histograms")
plt.show()

# --- Summary table (credible regions only; no p-values) ---
print("\n" + "=" * 78)
print("POSTERIOR Delta(t) SUMMARY — regions where the 95% CrI EXCLUDES ZERO")
print("=" * 78)
rows_delta = []
for font in fonts_list:
    R = results_common[font]
    cred = R['credible']
    regions, in_r, start = [], False, None
    for j in range(n_tc):
        if cred[j] and not in_r:
            start = t_ms_c[j]; in_r = True
        elif not cred[j] and in_r:
            regions.append((start, t_ms_c[j-1])); in_r = False
    if in_r:
        regions.append((start, t_ms_c[-1]))

    print(f"\n  {font}:")
    if not regions:
        print("    (none — the 95% CrI includes zero at every time point)")
    for s, e in regions:
        sl = slice(int(s), int(e) + 1)
        print(f"    {s:.0f}-{e:.0f} ms: mean Delta = {R['mean'][sl].mean():+.3f} items")
    print(f"    Credible points: {int(cred.sum())} / {n_tc}"
          f"   |  max P(Delta>0) = {R['p_dir'].max():.3f}")

    # Point summaries at canonical times for the manuscript
    for t_ms in [50, 100, 200]:
        j = int(t_ms)
        rows_delta.append({
            'Font': font, 't_ms': t_ms,
            'Delta_mean': R['mean'][j], 'CrI_low': R['lo'][j], 'CrI_high': R['hi'][j],
            'P_gt_0': R['p_dir'][j],
        })

df_delta_points = pd.DataFrame(rows_delta)
print("\n=== Delta(t) at canonical exposure times (for Results text) ===")
print(df_delta_points.to_string(index=False, float_format='{:.3f}'.format))
df_delta_points.to_csv(TABLE_DIR / "posterior_delta_canonical_times.csv", index=False)

### 4.4 Figure 6 — forest plot of population-level posteriors

In [ ]:
# 4.4 Figure 6: forest plot of population-level posterior means and 95% CrI
# ============================================================
from matplotlib.ticker import MaxNLocator

rows_forest = []
for font in fonts_list:
    for group in groups_list:
        K, C, t0 = get_KCt0(font, group)
        for par_name, arr in [("K", K), ("C", C), ("t0", t0)]:
            mean = np.mean(arr)
            lo, hi = np.quantile(arr, [0.025, 0.975])
            rows_forest.append({
                "font": font, "group": group,
                "parameter": par_name,
                "mean": float(mean), "ci_low": float(lo), "ci_high": float(hi),
                "label": f"{font} ({group})"
            })

df_forest = pd.DataFrame(rows_forest)
label_order = [f"{f} ({g})" for f in fonts_list for g in groups_list]

group_colors = {"Novice": "#1f77b4", "Expert": "#d62728"}

param_order = ["K", "C", "t0"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False)

for ax, par in zip(axes, param_order):
    dp = df_forest[df_forest["parameter"] == par]
    y_pos = {lbl: i for i, lbl in enumerate(label_order)}

    for _, row in dp.iterrows():
        y = y_pos[row["label"]]
        color = group_colors[row["group"]]
        if row["parameter"] != "t0":
            ax.hlines(y, row["ci_low"], row["ci_high"], colors=color, linewidth=3, alpha=0.7)
            ax.scatter(row["mean"], y, color=color, s=60, zorder=3,
                    edgecolors='black', linewidth=0.5)
            ax.set_yticks(range(len(label_order)), label_order)
            ax.set_xticks(dp[["ci_low", "mean", "ci_high"]].values.flatten())
            ax.tick_params(axis='x', rotation=90)
            ax.grid(axis="x", alpha=0.3, linestyle='--')
            ax.set_title(f"{par}", fontsize=13, fontweight='bold')
            if row["parameter"]=="K":
                ax.set_xlabel("Value (No. Letters)", labelpad=15)
            else:
                ax.set_xlabel("Value (Letters/sec)", labelpad=15)
        else:
            ax.hlines(y, row["ci_low"]*1000, row["ci_high"]*1000, colors=color, linewidth=3, alpha=0.7)
            ax.scatter(row["mean"]*1000, y, color=color, s=60, zorder=3,
                    edgecolors='black', linewidth=0.5)
            ax.set_yticks(range(len(label_order)), label_order)
            ticks = dp[["ci_low","mean","ci_high"]].values.flatten()*1000
            ax.set_xticks(ticks, minor=True)
            ax.grid(axis='x', which='minor', linestyle='--', alpha=0.3)
            ax.xaxis.set_major_locator(MaxNLocator(20))
            # ax.set_xticks(dp[["ci_low", "mean", "ci_high"]].values.flatten()*1e3)
            ax.tick_params(axis='x', rotation=90)
            ax.set_title(f"{par}", fontsize=13, fontweight='bold')
            ax.set_xlabel("Value (ms)", labelpad=15)

fig.suptitle("Posterior Means & 95% CrI", y=1.02, fontsize=14)
fig.tight_layout()
savefig("Fig06_forest_plot_population_posterior")
plt.show()

### 4.5 Table 2 — posterior contrasts (Expert − Novice)

In [ ]:
# 4.5 Table 2: posterior contrasts (Expert - Novice); t0 rows are in seconds (x1000 for ms)
# ============================================================

def contrast_summary(arr, label, alpha=0.05):
    mean = np.mean(arr)
    sd   = np.std(arr)
    lo   = np.quantile(arr, alpha / 2)
    hi   = np.quantile(arr, 1 - alpha / 2)
    p_gt0 = np.mean(arr > 0)
    return {"contrast": label, "mean": mean, "sd": sd,
            "ci_2.5%": lo, "ci_97.5%": hi, "P(>0)": p_gt0}

params = {}
for font in fonts_list:
    for group in groups_list:
        K, C, t0 = get_KCt0(font, group)
        params[(font, group)] = {"K": K, "C": C, "t0": t0}

rows = []
for par in ["K", "C", "t0"]:
    diff_helv = params[("Helvetica", "Expert")][par] - params[("Helvetica", "Novice")][par]
    rows.append(contrast_summary(diff_helv, f"{par}: Expert-Novice (Helvetica)"))

    diff_klim = params[("Klimax Plus", "Expert")][par] - params[("Klimax Plus", "Novice")][par]
    rows.append(contrast_summary(diff_klim, f"{par}: Expert-Novice (Klimax Plus)"))

    avg_expert = 0.5 * (params[("Helvetica", "Expert")][par] + params[("Klimax Plus", "Expert")][par])
    avg_novice = 0.5 * (params[("Helvetica", "Novice")][par] + params[("Klimax Plus", "Novice")][par])
    rows.append(contrast_summary(avg_expert - avg_novice, f"{par}: Expert-Novice (averaged)"))

    interaction = diff_klim - diff_helv
    rows.append(contrast_summary(interaction, f"{par}: Interaction (Klimax Plus - Helvetica Expert advantage)"))

df_contrasts = pd.DataFrame(rows)
print("=== Posterior Contrasts: Expert vs Novice ===")
print(df_contrasts.to_string(index=False, float_format='{:.4f}'.format))
df_contrasts.to_csv(TABLE_DIR / "Table2_posterior_contrasts.csv", index=False)

### 4.6 Figure 5 — Expert − Novice posterior difference distributions

In [ ]:
# 4.6 Figure 5: Expert - Novice posterior difference distributions
# ============================================================

font_colors = {"Helvetica": "#1b7837", "Klimax Plus": "#f28e2b"}   # green / orange, distinguishable for color-blind readers

# parameter -> (symbol, x-axis label with units, scale factor)
par_specs = {
    "K":  (r"$K$",   r"$\Delta K$ (No. Letters)", 1.0),
    "C":  (r"$C$",   r"$\Delta C$ (letters/s)",   1.0),
    "t0": (r"$t_0$", r"$\Delta t_0$ (ms)",        1000.0),   # t0 is in seconds → convert to ms
}

# Common x-scale within each row, so the two fonts (e.g., the two K panels) can be compared directly
fig, axes = plt.subplots(3, 2, figsize=(14, 14), sharex='row')

for row, par in enumerate(["K", "C", "t0"]):
    sym, xlab, scale = par_specs[par]
    for col, font in enumerate(fonts_list):
        ax = axes[row, col]
        diff = scale * (params[(font, "Expert")][par] - params[(font, "Novice")][par])
        mean_diff = np.mean(diff)
        p = np.mean(diff > 0)

        ax.hist(diff, bins=80, density=True, alpha=0.55, color=font_colors[font], edgecolor='none')
        ax.axvline(0, color='red', lw=1.5, ls='--', alpha=0.7)    # zero: no group difference
        ax.axvline(mean_diff, color='black', lw=2, ls='-')           # posterior mean

        ax.set_title(rf"{sym}: Expert $-$ Novice ({font})" "\n"
                     rf"mean = ${mean_diff:.3f}$, $P(\Delta > 0) = {p:.3f}$", fontsize=10)
        ax.set_xlabel(xlab)
        ax.set_ylabel("Density")
        ax.grid(alpha=0.2)

fig.suptitle("Expert $-$ Novice: Posterior Differences", fontsize=15, y=1.01)
fig.tight_layout()
savefig("Fig05_expert_novice_posterior_differences")
plt.show()

### 4.7 Figure 7 — font effect (Klimax Plus − Helvetica) by group

In [ ]:
# 4.7 Figure 7: font effect (Klimax Plus - Helvetica) by group
# ============================================================

group_colors = {"Novice": "#1f77b4", "Expert": "#d62728"}
x_labels = ["K (No. letters)", "C (Letters/sec)", "t0 (msec)"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col_idx, (par, xlab) in enumerate(zip(["K", "C", "t0"], x_labels)):
    ax = axes[col_idx]
    diff_novice = params[("Klimax Plus", "Novice")][par] - params[("Helvetica", "Novice")][par]
    diff_expert = params[("Klimax Plus", "Expert")][par] - params[("Helvetica", "Expert")][par]
    if par == "t0":                       # t0 is in seconds → convert to msec
        diff_novice = diff_novice * 1000
        diff_expert = diff_expert * 1000

    ax.hist(diff_novice, bins=80, density=True, alpha=0.5, color=group_colors["Novice"],
            label=f"Novice (P>0={np.mean(diff_novice > 0):.2f})", edgecolor='none')
    ax.hist(diff_expert, bins=80, density=True, alpha=0.5, color=group_colors["Expert"],
            label=f"Expert (P>0={np.mean(diff_expert > 0):.2f})", edgecolor='none')
    ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.7)

    ax.set_title(f"{par}: Klimax Plus - Helvetica")
    ax.set_xlabel(xlab)
    if col_idx == 0:
        ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle("Font Effect (Klimax Plus - Helvetica) by Group", fontsize=14, y=1.01)
fig.tight_layout()
savefig("Fig07_font_effect_by_group")
plt.show()

### 4.8 Random-effect standard deviations and group means

In [ ]:
# 4.8 Random-effect standard deviations and group means
# ============================================================

print("=== Random slope SDs for font effect ===")
for par in ['sd_font_klin', 'sd_font_clin', 'sd_font_tlin']:
    vals = _extract_1d(post_full, par)
    print(f"  {par}: mean={np.mean(vals):.4f}, 95% CrI=[{np.quantile(vals, 0.025):.4f}, {np.quantile(vals, 0.975):.4f}]")

print("\n=== Participant RE SDs ===")
for par in ['sd_klin_part', 'sd_clin_part', 'sd_tlin_part']:
    vals = _extract_1d(post_full, par)
    print(f"  {par}: mean={np.mean(vals):.4f}, 95% CrI=[{np.quantile(vals, 0.025):.4f}, {np.quantile(vals, 0.975):.4f}]")

print("\n=== Group means on transformed scale ===")
for par, transform, label in [
    ('group_mean_klin', lambda x: 5*sigmoid(x), 'K (items)'),
    ('group_mean_clin', np.exp, 'C (items/s)'),
    ('group_mean_tlin', lambda x: np.exp(x)*1000, 't0 (ms)')
]:
    vals = _extract_vec(post_full, par)
    for g, gname in enumerate(['Novice', 'Expert']):
        transformed = transform(vals[:, g])
        print(f"  {label} [{gname}]: mean={np.mean(transformed):.3f}, "
              f"95% CrI=[{np.quantile(transformed, 0.025):.3f}, {np.quantile(transformed, 0.975):.3f}]")

### 4.9 Group and font effects on the natural and latent scales

In [ ]:
# 4.9 Group effects on the natural scale; font and interaction effects on the latent scale
# ============================================================

print("=== Group Effects (Expert - Novice) on natural scale ===")
for par, transform, label, unit in [
    ('group_mean_klin', lambda x: 5*sigmoid(x), 'ΔK', 'items'),
    ('group_mean_clin', np.exp, 'ΔC', 'items/s'),
    ('group_mean_tlin', lambda x: np.exp(x)*1000, 'Δt0', 'ms')
]:
    vals = _extract_vec(post_full, par)
    expert = transform(vals[:, 1])
    novice = transform(vals[:, 0])
    diff = expert - novice
    print(f"  {label}: mean={np.mean(diff):.3f} {unit}, "
          f"95% CrI=[{np.quantile(diff, 0.025):.3f}, {np.quantile(diff, 0.975):.3f}], "
          f"P(>0)={np.mean(diff > 0):.3f}")

print("\n=== Font Population Effects (Klimax Plus - Helvetica) on latent scale ===")
for par in ['font_klin', 'font_clin', 'font_tlin']:
    vals = _extract_1d(post_full, par)
    print(f"  {par}: mean={np.mean(vals):.4f}, "
          f"95% CrI=[{np.quantile(vals, 0.025):.4f}, {np.quantile(vals, 0.975):.4f}], "
          f"P(>0)={np.mean(vals > 0):.3f}")

print("\n=== Font × Group Interaction (latent scale) ===")
for par in ['fontgroup_klin', 'fontgroup_clin', 'fontgroup_tlin']:
    vals = _extract_1d(post_full, par)
    print(f"  {par}: mean={np.mean(vals):.4f}, "
          f"95% CrI=[{np.quantile(vals, 0.025):.4f}, {np.quantile(vals, 0.975):.4f}], "
          f"P(>0)={np.mean(vals > 0):.3f}")

## 5. Hypothesis tests

- **H1** — Expert vs. Novice performance overall and per font (Sections 5 and 4).
- **H2** — Expert vs. Novice per exposure time (Sections 7, 4.3 and 8).
- **H3** — Impact of age on performance; does expertise moderate age effects? (Sections 6 and 8).

### 5.1 Participant-level summaries


In [ ]:
# 5.1 Participant-level summaries (age merged; population-level K, C, t0 attached)
# ============================================================

# Merge age into trial-level data
dat_age = dat.copy()
print(f"Age data present: {dat_age['age'].notna().sum()} / {len(dat_age)} trials have age")

# Compute participant-level posterior K, C, t0 from the Bayesian model
part_params = []
for font in fonts_list:
    for group in groups_list:
        K, C, t0 = get_KCt0(font, group)
        # Population-level posterior means
        K_mean, C_mean, t0_mean = K.mean(), C.mean(), t0.mean()

        # Get participant-level means from empirical data
        sub = dat_age[(dat_age['font']==font) & (dat_age['group']==group)]
        for pid in sub['participant'].unique():
            p_sub = sub[sub['participant']==pid]
            age = p_sub['age'].iloc[0] if p_sub['age'].notna().any() else np.nan
            part_params.append({
                'participant': pid, 'group': group, 'font': font,
                'age': age,
                'mean_P': p_sub['P'].mean(),
                'K_pop': K_mean, 'C_pop': C_mean, 't0_pop': t0_mean,
            })

df_part = pd.DataFrame(part_params)

# Also create participant-level summary collapsing over font
df_part_grand = (
    dat_age.groupby(['participant', 'group'])
    .agg(mean_P=('P', 'mean'), age=('age', 'first'))
    .reset_index()
)

# For font-level
df_part_font = (
    dat_age.groupby(['participant', 'group', 'font'])
    .agg(mean_P=('P', 'mean'), age=('age', 'first'))
    .reset_index()
)

print(f"\nParticipant-level data: {len(df_part)} rows")
display(df_part.head(10))

### 5.2 t-tests on hierarchical (partially pooled) per-participant parameters — descriptive

In [ ]:
# 5.2 t-tests on hierarchical per-participant TVA parameters - Expert vs Novice per font
# ============================================================
# Expert and Novice are DIFFERENT people (between-subjects), so these are
# independent-samples t-tests, not paired.
#
#  * Multiplicity: six tests are run over the same participants, so
#    Holm-Bonferroni and Benjamini-Hochberg FDR adjusted p-values are reported
#    alongside the raw ones.
#  * Circularity: the per-participant K/C/t0 used here are reconstructed from
#    the hierarchical model, which ALREADY contains Font x Group fixed effects.
#    Partial pooling shrinks each participant toward their group x font mean,
#    which shrinks within-group variance and can inflate d / deflate p. These
#    tests are therefore DESCRIPTIVE. The primary group inference is the
#    posterior contrast set (Section 4.5); the independent, non-hierarchical
#    per-participant TVA fits in Section 9.3 provide the unshrunk estimates
#    reported in the manuscript (Table 3, Section 9.7).

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

print("=" * 78)
print("T-TESTS: Expert vs Novice — TVA parameters per Font  (SECONDARY analysis)")
print("  Parameters are hierarchical-model posterior means -> shrunk. See Section 9.3.")
print("=" * 78)

# Extract participant-level posterior K, C, t0 from the hierarchical model
# part_klin, part_clin, part_tlin are Deterministic nodes in the model
post = idata_full.posterior

part_klin_post = post['part_klin'].mean(dim=['chain', 'draw']).values  # (n_participants,)
part_clin_post = post['part_clin'].mean(dim=['chain', 'draw']).values
part_tlin_post = post['part_tlin'].mean(dim=['chain', 'draw']).values

# Also need font effects per participant
font_klin_mean = float(post['font_klin'].mean())
font_clin_mean = float(post['font_clin'].mean())
font_tlin_mean = float(post['font_tlin'].mean())

sd_font_klin_mean = float(post['sd_font_klin'].mean())
sd_font_clin_mean = float(post['sd_font_clin'].mean())
sd_font_tlin_mean = float(post['sd_font_tlin'].mean())

z_font_klin_post = post['z_font_klin'].mean(dim=['chain', 'draw']).values
z_font_clin_post = post['z_font_clin'].mean(dim=['chain', 'draw']).values
z_font_tlin_post = post['z_font_tlin'].mean(dim=['chain', 'draw']).values

fg_klin = float(post['fontgroup_klin'].mean())
fg_clin = float(post['fontgroup_clin'].mean())
fg_tlin = float(post['fontgroup_tlin'].mean())

# Build participant x font K, C, t0 estimates
rows_kct = []
for pi, pname in enumerate(participant_names):
    grp_idx = int(participant_group[pi])
    grp_name = 'Expert' if grp_idx == 1 else 'Novice'
    is_expert = grp_idx

    for font_name, isK in [('Helvetica', 0), ('Klimax Plus', 1)]:
        klin_i = part_klin_post[pi] + (font_klin_mean + sd_font_klin_mean * z_font_klin_post[pi]) * isK + fg_klin * isK * is_expert
        clin_i = part_clin_post[pi] + (font_clin_mean + sd_font_clin_mean * z_font_clin_post[pi]) * isK + fg_clin * isK * is_expert
        tlin_i = part_tlin_post[pi] + (font_tlin_mean + sd_font_tlin_mean * z_font_tlin_post[pi]) * isK + fg_tlin * isK * is_expert

        K_i = 5.0 / (1 + np.exp(-klin_i))
        C_i = np.exp(clin_i)
        t0_i = np.exp(tlin_i)

        age_row = participant_info[participant_info['Participants_ID'] == pname]
        age_val = int(age_row['age'].iloc[0]) if len(age_row) > 0 else np.nan

        rows_kct.append({
            'participant': pname, 'group': grp_name, 'font': font_name,
            'K': K_i, 'C': C_i, 't0': t0_i, 'age': age_val
        })

df_kct = pd.DataFrame(rows_kct)
print("\nParticipant-level TVA parameters (hierarchical posterior point estimates):")
display(df_kct.head(8))

# --- T-tests per Font x Parameter ---
results_ttest = []
for font in fonts_list:
    for par in ['K', 'C', 't0']:
        nov = df_kct[(df_kct['font'] == font) & (df_kct['group'] == 'Novice')][par].values
        exp = df_kct[(df_kct['font'] == font) & (df_kct['group'] == 'Expert')][par].values
        t_stat, p_val = ttest_ind(exp, nov)
        d = pg.compute_effsize(exp, nov, eftype='cohen')
        diff = exp.mean() - nov.mean()
        se = np.sqrt(np.var(exp) / len(exp) + np.var(nov) / len(nov))
        ci_lo = diff - 1.96 * se
        ci_hi = diff + 1.96 * se
        df_val = len(exp) + len(nov) - 2

        results_ttest.append({
            'Font': font, 'Parameter': par,
            'Mean_Novice': nov.mean(), 'Mean_Expert': exp.mean(),
            'Mean_Diff': diff, 'CI_low': ci_lo, 'CI_high': ci_hi,
            't_stat': t_stat, 'df': df_val, 'p_value': p_val,
            'Cohens_d': d,
        })

df_ttest = pd.DataFrame(results_ttest)

# ---- Correct for the six comparisons ----
p_raw = df_ttest['p_value'].values
_, p_holm, _, _ = multipletests(p_raw, alpha=0.05, method='holm')
_, p_bh,   _, _ = multipletests(p_raw, alpha=0.05, method='fdr_bh')
df_ttest['p_holm'] = p_holm
df_ttest['p_BH']   = p_bh


def _stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


df_ttest['Sig']      = [_stars(p) for p in p_raw]     # uncorrected (kept for figures)
df_ttest['Sig_holm'] = [_stars(p) for p in p_holm]
df_ttest['Sig_BH']   = [_stars(p) for p in p_bh]

print("\n=== Results, raw and multiplicity-corrected (family of 6 tests) ===")
for _, r in df_ttest.iterrows():
    fs = 'Helvetica' if 'Helv' in r['Font'] else 'Klimax Plus'
    print(f"  {fs:>9s} {r['Parameter']:>2s}: t({int(r['df'])})={r['t_stat']:+.3f}, "
          f"p={r['p_value']:.4f} {r['Sig']:>3s} | "
          f"p_holm={r['p_holm']:.4f} {r['Sig_holm']:>3s} | "
          f"p_BH={r['p_BH']:.4f} {r['Sig_BH']:>3s} | d={r['Cohens_d']:+.3f}")

print("\n  NOTE: only tests that survive Holm/BH should be described as reliable.")
print("  Everything else is exploratory and must lean on the Bayesian contrasts.")

print("\n=== Full Results Table ===")
display(df_ttest.round(4))
df_ttest.to_csv(TABLE_DIR / "hierarchical_parameter_ttests.csv", index=False)

### 5.3 Figure 3 — TVA parameters by Font × Group (hierarchical estimates); Figure S3

In [ ]:
# 5.3 Figure 3: TVA parameters by Font x Group (hierarchical estimates); Figure S3
# ============================================================

group_colors = {'Novice': '#1f77b4', 'Expert': '#d62728'}

# --- Figure 3: box + strip plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
par_list = ['K', 'C', 't0']   # (`params` is the posterior dict from Section 4.5)
ylabels = ['K (VSTM Capacity)', 'C (Processing Speed, elem/s)', 't₀ (Perceptual Threshold, s)']

for ax, par, ylab in zip(axes, par_list, ylabels):
    positions = []
    for fi, font in enumerate(fonts_list):
        for gi, group in enumerate(groups_list):
            x = fi * 2.5 + gi * 1.0
            positions.append(x)
            sub = df_kct[(df_kct['font']==font) & (df_kct['group']==group)][par]
            bp = ax.boxplot([sub], positions=[x], widths=0.7,
                           patch_artist=True, showmeans=True,
                           meanprops=dict(marker='D', markerfacecolor='white', markersize=6))
            bp['boxes'][0].set_facecolor(group_colors[group])
            bp['boxes'][0].set_alpha(0.5)
            # Overlay individual points
            ax.scatter(np.random.normal(x, 0.1, len(sub)), sub,
                      color=group_colors[group], alpha=0.6, s=20, zorder=3)

    # Add significance brackets
    for fi, font in enumerate(fonts_list):
        row = df_ttest[(df_ttest['Font']==font) & (df_ttest['Parameter']==par)].iloc[0]
        x1, x2 = fi * 2.5, fi * 2.5 + 1.0
        y_max = df_kct[df_kct['font']==font][par].max()
        y_bar = y_max * 1.08
        ax.plot([x1, x1, x2, x2], [y_bar*0.98, y_bar, y_bar, y_bar*0.98], 'k-', lw=1)
        ax.text((x1+x2)/2, y_bar*1.01, f"p={row['p_value']:.4f} {row['Sig']}",
                ha='center', fontsize=8)

    ax.set_xticks([fi * 2.5 + 0.5 for fi in range(len(fonts_list))])
    ax.set_xticklabels(list(fonts_list))
    ax.set_ylabel(ylab)
    ax.set_title(par, fontsize=14, fontweight='bold')
    ax.grid(alpha=0.2, axis='y')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=group_colors['Novice'], alpha=0.5, label='Novice'),
                   Patch(facecolor=group_colors['Expert'], alpha=0.5, label='Expert')]
axes[2].legend(handles=legend_elements, loc='best', fontsize=10)

fig.suptitle(f'TVA Parameters: Novice vs Expert (N={n_novice}+{n_expert})',
             fontsize=16, y=1.02)
fig.tight_layout()
savefig("Fig03_tva_parameters_hierarchical")
plt.show()

# --- Figure S3: Expert - Novice differences by font, box + strip ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
font_colors = {'Helvetica': '#1f77b4', 'Klimax Plus': '#d62728'}

for ax, par, ylab in zip(axes, par_list, [f'Δ{p} (Expert−Novice)' for p in par_list]):
    for fi, font in enumerate(fonts_list):
        exp_vals = df_kct[(df_kct['font']==font) & (df_kct['group']=='Expert')][par].values
        nov_vals = df_kct[(df_kct['font']==font) & (df_kct['group']=='Novice')][par].values
        # For between-subjects: difference of group means per bootstrap isn't "paired"
        # Show individual values as strip
        diff_exp_mean = exp_vals - nov_vals.mean()  # each expert vs novice mean
        row = df_ttest[(df_ttest['Font']==font) & (df_ttest['Parameter']==par)].iloc[0]

        bp = ax.boxplot([exp_vals - nov_vals.mean()], positions=[fi], widths=0.6,
                       patch_artist=True, showmeans=True,
                       meanprops=dict(marker='D', markerfacecolor='white', markersize=6))
        bp['boxes'][0].set_facecolor(font_colors[font])
        bp['boxes'][0].set_alpha(0.5)
        ax.scatter(np.random.normal(fi, 0.08, len(exp_vals)), diff_exp_mean,
                  color=font_colors[font], alpha=0.5, s=25, zorder=3)

        sig_txt = f"p={row['p_value']:.4f}\n{row['Sig']}"
        ax.text(fi, ax.get_ylim()[1] if ax.get_ylim()[1] != 1 else exp_vals.max()*1.1,
                sig_txt, ha='center', fontsize=9, color=font_colors[font], fontweight='bold')

    ax.axhline(0, color='grey', ls='--', lw=1)
    ax.set_xticks(range(len(fonts_list)))
    ax.set_xticklabels(list(fonts_list))
    ax.set_ylabel(ylab)
    ax.set_title(par, fontsize=13, fontweight='bold')
    ax.grid(alpha=0.2, axis='y')

fig.suptitle('Expert − Novice Differences by Font', fontsize=15, y=1.02)
fig.tight_layout()
savefig("FigS03_expert_novice_differences_by_font")
plt.show()

### 5.4 Figure S4 — posterior difference violin plots

In [ ]:
# 5.4 Figure S4: posterior difference violin plots
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
par_names = ['K', 'C', 't0']
font_colors_list = ['#1f77b4', '#d62728']

for ax, par_name in zip(axes, par_names):
    data_violins = []
    labels = []
    for fi, font in enumerate(fonts_list):
        K_e, C_e, t0_e = get_KCt0(font, "Expert")
        K_n, C_n, t0_n = get_KCt0(font, "Novice")

        if par_name == 'K': delta = K_e - K_n
        elif par_name == 'C': delta = C_e - C_n
        else: delta = t0_e - t0_n

        data_violins.append(delta)
        labels.append(font)

        # Compute HDI
        lo, hi = np.quantile(delta, [0.025, 0.975])
        p_gt0 = (delta > 0).mean()

    parts = ax.violinplot(data_violins, positions=range(len(fonts_list)),
                          showmeans=False, showmedians=False, showextrema=False)
    for fi, pc in enumerate(parts['bodies']):
        pc.set_facecolor(font_colors_list[fi])
        pc.set_alpha(0.5)

    for fi in range(len(fonts_list)):
        d = data_violins[fi]
        lo, hi = np.quantile(d, [0.025, 0.975])
        med = np.median(d)
        ax.scatter(fi, med, color='white', edgecolor='black', s=40, zorder=5)
        ax.vlines(fi, lo, hi, color='grey', lw=2)

        p_gt0 = (d > 0).mean()
        ax.text(fi + 0.35, med, f'95%HDI\n[{lo:.3f}, {hi:.3f}]\nP(d>0)={p_gt0:.3f}',
                fontsize=7, va='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.axhline(0, color='grey', ls='--', lw=1)
    ax.set_xticks(range(len(fonts_list)))
    ax.set_xticklabels(labels)
    ax.set_ylabel(f'Δ{par_name} (Expert − Novice)')
    ax.set_title(f'Posterior Difference: {par_name}', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.2, axis='y')

fig.suptitle('Bayesian Posterior Differences: Expert − Novice', fontsize=15, y=1.02)
fig.tight_layout()
savefig("FigS04_posterior_difference_violins")
plt.show()

### 5.5 Mixed ANOVA, linear mixed models and Tukey HSD (Table 4)

In [ ]:
# 5.5 Mixed ANOVA, linear mixed models and Tukey HSD post-hoc tests (Table 4)
# ============================================================
# Sign convention: statsmodels' pairwise_tukeyhsd reports
#     meandiff = mean(group2) - mean(group1)
# i.e. the SECOND column minus the FIRST. The Tukey table is therefore printed
# with an explicit Delta M = Condition_2 - Condition_1 header, the cell means,
# and a plain-language direction column so that the sign cannot be misread.
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print("=" * 78)
print("MIXED ANOVA: Group (between) x Font (within)")
print("=" * 78)

aov_font = pg.mixed_anova(
    data=df_part_font, dv='mean_P', within='font',
    between='group', subject='participant'
)
display(aov_font)
aov_font.to_csv(TABLE_DIR / "mixed_anova_group_by_font.csv", index=False)
##################################################################
# --- Font x Group interaction on raw scores (reported for the buffering claim) ---
inter = aov_font.loc[aov_font['Source'] == 'Interaction'].iloc[0]


def _col(df, *names):
    """First of `names` that is a column of `df` (pingouin renamed its columns in 0.6+)."""
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(names)


wide = df_part_font.pivot_table(index=['participant', 'group'], columns='font',
                                values='mean_P').reset_index()
wide['penalty'] = wide['Helvetica'] - wide['Klimax Plus']   # font penalty per participant

pen = wide.groupby('group')['penalty'].agg(['mean', 'std', 'count'])
print("\n--- Font penalty (Helvetica - Klimax Plus), letters correct ---")
print(pen.round(3).to_string())

# In a 2x2 mixed design, this independent t-test on penalty scores IS the
# interaction test (t^2 = F); it adds the CI, effect size and Bayes factor.
tt = pg.ttest(wide.loc[wide['group'] == 'Novice', 'penalty'],
              wide.loc[wide['group'] == 'Expert', 'penalty'],
              paired=False, correction=False)
diff = pen.loc['Novice', 'mean'] - pen.loc['Expert', 'mean']
ci_lo, ci_hi = tt[_col(tt, 'CI95%', 'CI95')].iloc[0]

print(f"\nFont x Group interaction: F(1, {int(inter['DF2'])}) = {inter['F']:.2f}, "
      f"p = {inter[_col(aov_font, 'p-unc', 'p_unc')]:.3f}, partial eta^2 = {inter['np2']:.3f}")
print(f"Penalty difference (Novice - Expert) = {diff:.3f} letters, "
      f"95% CI [{ci_lo:.2f}, {ci_hi:.2f}], d = {tt[_col(tt, 'cohen-d', 'cohen_d')].iloc[0]:.2f}, "
      f"BF10 = {float(tt['BF10'].iloc[0]):.2f}")
print(f"Check: t^2 = {tt['T'].iloc[0]**2:.3f} equals F = {inter['F']:.3f}")

##################################################################
# --- LMM: P ~ group * font + (1|participant) ---
# group and font are string columns, so patsy codes them as categorical factors
# (treatment coding; reference levels Expert and Helvetica).
print("\n" + "=" * 78)
print("LINEAR MIXED MODEL: P ~ group x font + (1|participant)")
print("=" * 78)

dat_lmm = dat_age.copy()
lmm1 = smf.mixedlm("P ~ group * font", dat_lmm, groups=dat_lmm["participant"]).fit(reml=True)
print(lmm1.summary())

# --- LMM with age: P ~ group * font * age + (1|participant) ---
print("\n" + "=" * 78)
print("LINEAR MIXED MODEL: P ~ group x font x age + (1|participant)")
print("=" * 78)

dat_lmm['age_range'] = pd.cut(dat_lmm['age'], bins=[20, 35, 50, 70],
                              labels=['young', 'middle', 'senior'])

lmm2 = smf.mixedlm("P ~ group * font * age", dat_lmm, groups=dat_lmm["participant"]).fit(reml=True)
print(lmm2.summary())

# --- LMM with age_range ---
print("\n" + "=" * 78)
print("LINEAR MIXED MODEL: P ~ group x font x age_range + (1|participant)")
print("=" * 78)

lmm3 = smf.mixedlm("P ~ group * font * age_range", dat_lmm,
                   groups=dat_lmm["participant"]).fit(reml=True)
print(lmm3.summary())

# ================================================================
# TUKEY POST-HOC — with unambiguous sign convention
# ================================================================
print("\n" + "=" * 78)
print("TUKEY HSD: Font x Group (4 conditions), outcome = mean letters correct")
print("=" * 78)

df_tukey = df_part_font.copy()
# Compact tag for the Tukey condition labels ('Klimax' abbreviates 'Klimax Plus'
# inside the underscore-joined condition names).
KLIMAX_TAG = 'Klimax'
df_tukey['condition'] = (df_tukey['group'] + '_' +
                         df_tukey['font'].str.replace('Klimax Plus', KLIMAX_TAG))

# Cell means, printed first so the direction of every contrast is verifiable
cond_means = df_tukey.groupby('condition')['mean_P'].agg(['mean', 'std', 'count'])
print("\n--- Cell means (mean letters correct per participant, averaged over exposures) ---")
print(cond_means.round(4).to_string())

tukey = pairwise_tukeyhsd(df_tukey['mean_P'], df_tukey['condition'], alpha=0.05)
print("\n--- Raw statsmodels output (meandiff = group2 - group1) ---")
print(tukey)

# --- Self-documenting version -------------------------------------------------
tuk_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
tuk_df = tuk_df.rename(columns={'group1': 'Condition_1', 'group2': 'Condition_2',
                                'meandiff': 'DeltaM_C2_minus_C1'})
tuk_df['DeltaM_C2_minus_C1'] = tuk_df['DeltaM_C2_minus_C1'].astype(float)
tuk_df['p-adj'] = tuk_df['p-adj'].astype(float)
tuk_df['lower'] = tuk_df['lower'].astype(float)
tuk_df['upper'] = tuk_df['upper'].astype(float)

means_map = cond_means['mean'].to_dict()
tuk_df['Mean_C1'] = tuk_df['Condition_1'].map(means_map)
tuk_df['Mean_C2'] = tuk_df['Condition_2'].map(means_map)
tuk_df['Higher_scoring_condition'] = np.where(
    tuk_df['DeltaM_C2_minus_C1'] > 0, tuk_df['Condition_2'], tuk_df['Condition_1'])
tuk_df['Direction'] = [
    f"{r['Condition_2']} > {r['Condition_1']}" if r['DeltaM_C2_minus_C1'] > 0
    else f"{r['Condition_1']} > {r['Condition_2']}"
    for _, r in tuk_df.iterrows()
]

print("\n--- Manuscript-ready table (Delta M = Condition_2 - Condition_1) ---")
print(tuk_df[['Condition_1', 'Condition_2', 'Mean_C1', 'Mean_C2',
              'DeltaM_C2_minus_C1', 'p-adj', 'lower', 'upper', 'reject',
              'Direction']].to_string(index=False, float_format='{:.4f}'.format))
tuk_df.to_csv(TABLE_DIR / "Table4_tukey_hsd.csv", index=False)

# LaTeX rows for the manuscript table
print("\n--- LaTeX body rows (copy into the manuscript table) ---")
for _, r in tuk_df.iterrows():
    c1 = r['Condition_1'].replace('_', r'\_')
    c2 = r['Condition_2'].replace('_', r'\_')
    rej = 'Yes' if r['reject'] else 'No'
    print(f"{c1} & {c2} & ${r['DeltaM_C2_minus_C1']:+.3f}$ & {r['p-adj']:.3f} & "
          f"[{r['lower']:+.2f}, {r['upper']:+.2f}] & {rej} \\\\")

# --- Interpretation guard rails ----------------------------------------------
print("\n" + "-" * 78)
print("INTERPRETATION CAVEATS for the Tukey table")
print("-" * 78)
key = tuk_df[(tuk_df['Condition_1'] == 'Expert_Klimax') &
             (tuk_df['Condition_2'] == 'Novice_Helvetica')]
if len(key):
    k = key.iloc[0]
    print(f"  * Expert_Klimax vs Novice_Helvetica: Delta M = {k['DeltaM_C2_minus_C1']:+.3f} "
          f"(p_adj = {k['p-adj']:.3f}) -> {k['Direction']}.")
    print("    i.e. novices reading the EASY font outscored experts reading the HARD")
    print("    font. This is the expected ordering (the font effect is much larger")
    print("    than the expertise effect) and must NOT be described as experts")
    print("    beating novices across fonts.")
print("  * The outcome is mean letters correct collapsed over exposure duration.")
print("    The two groups' exposure grids differ (Novice 16.67-216.67 ms vs")
print("    Expert 10-200 ms), so BETWEEN-GROUP cells are not exposure-matched and")
print("    the between-group contrasts are confounded with the timing fault.")
print("    Only the two WITHIN-GROUP font contrasts (Expert_Helvetica vs")
print("    Expert_Klimax; Novice_Helvetica vs Novice_Klimax) are unconfounded.")
print("  * Duration-matched group comparisons are in Section 7.2; the model-based")
print("    group comparison that accounts for exposure duration is the TVA fit.")

## 6. H3 — Age × expertise (Figure 11)

In [ ]:
# 6. H3: Age x Expertise analysis (Figure 11)
# ============================================================
# With p_group x age = .750 (Section 8, D4) the appropriate statement is
# "no evidence of a differential age-related decline", which is how the figure
# titles are worded. The oldest-expert / youngest-novice comparison aggregates
# over fonts first and then takes n = 3 participants per subgroup, matching
# the descriptive comparison in Section 8 (D3).

print("=" * 78)
print("H3: AGE x EXPERTISE ANALYSIS")
print("=" * 78)

# --- Scatter: age vs mean performance per participant ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Overall performance vs age
ax = axes[0]
for grp, color, marker in [('Novice', '#1f77b4', 's'), ('Expert', '#d62728', 'o')]:
    sub = df_part_grand[df_part_grand['group'] == grp]
    ax.scatter(sub['age'], sub['mean_P'], c=color, marker=marker, s=60, alpha=0.7, label=grp)
    z = np.polyfit(sub['age'], sub['mean_P'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['age'].min(), sub['age'].max(), 50)
    ax.plot(x_line, p(x_line), color=color, ls='--', lw=1.5, alpha=0.7)
    r, p_val = stats.pearsonr(sub['age'], sub['mean_P'])
    ax.text(0.05 if grp == 'Novice' else 0.55, 0.95 if grp == 'Novice' else 0.85,
            f'{grp}: r={r:.3f}, p={p_val:.3f}',
            transform=ax.transAxes, fontsize=9, color=color)
ax.set_xlabel('Age')
ax.set_ylabel('Mean Number Correct')
ax.set_title('Overall Performance vs Age')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Per font
ax = axes[1]
for grp, color in [('Novice', '#1f77b4'), ('Expert', '#d62728')]:
    for font, marker in [('Helvetica', 'o'), ('Klimax Plus', 's')]:
        sub = df_part_font[(df_part_font['group'] == grp) & (df_part_font['font'] == font)]
        label = f'{grp} ({font[:5]})'
        ax.scatter(sub['age'], sub['mean_P'], c=color, marker=marker,
                   s=40, alpha=0.6, label=label,
                   edgecolors='black' if font == 'Klimax Plus' else 'none', linewidth=0.5)
ax.set_xlabel('Age')
ax.set_ylabel('Mean Number Correct')
ax.set_title('Performance vs Age by Font x Group')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)

# Panel 3: TVA K parameter vs age
ax = axes[2]
for grp, color in [('Novice', '#1f77b4'), ('Expert', '#d62728')]:
    sub = df_kct[df_kct['group'] == grp].groupby('participant').agg(
        {'K': 'mean', 'age': 'first'}).reset_index()
    ax.scatter(sub['age'], sub['K'], c=color, s=60, alpha=0.7, label=grp)
    z = np.polyfit(sub['age'], sub['K'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['age'].min(), sub['age'].max(), 50)
    ax.plot(x_line, p(x_line), color=color, ls='--', lw=1.5, alpha=0.7)
    r, p_val = stats.pearsonr(sub['age'], sub['K'])
    ax.text(0.05 if grp == 'Novice' else 0.55, 0.95 if grp == 'Novice' else 0.85,
            f'{grp}: r={r:.3f}, p={p_val:.3f}',
            transform=ax.transAxes, fontsize=9, color=color)
ax.set_xlabel('Age')
ax.set_ylabel('K (VSTM Capacity)')
ax.set_title('K vs Age by Group')
ax.legend()
ax.grid(alpha=0.3)

fig.suptitle('H3: Age x Expertise — no evidence of a differential age-related decline '
             r'($p_{\mathrm{group \times age}} = .75$)', fontsize=14, y=1.03)
fig.tight_layout()
savefig("Fig11_age_expertise_scatter")
plt.show()

# --- Descriptive comparison: 3 oldest experts vs 3 youngest novices ---
# (aggregate over fonts FIRST so that n = 3 participants, not 3 participant-font rows)
print("\n--- Descriptive comparison (illustrative only, n = 3 per subgroup) ---")
_agg_kct = df_kct.groupby(['participant', 'group']).agg(
    {'K': 'mean', 'C': 'mean', 't0': 'mean', 'age': 'first'}).reset_index()

oldest_exp   = _agg_kct[_agg_kct['group'] == 'Expert'].nlargest(3, 'age')
youngest_nov = _agg_kct[_agg_kct['group'] == 'Novice'].nsmallest(3, 'age')

print("\n3 oldest Experts:")
display(oldest_exp[['participant', 'age', 'K', 'C', 't0']].round(3))
print("\n3 youngest Novices:")
display(youngest_nov[['participant', 'age', 'K', 'C', 't0']].round(3))
print("  NOTE: n = 3 per subgroup. Descriptive only — no inference is supported.")

# --- Age correlation table ---
print("\n--- Pearson correlations: Age x TVA parameters (uncorrected, exploratory) ---")
for grp in ['Novice', 'Expert']:
    print(f"\n  {grp}:")
    sub = df_kct[df_kct['group'] == grp].groupby('participant').agg(
        {'K': 'mean', 'C': 'mean', 't0': 'mean', 'age': 'first'}).reset_index()
    for par in ['K', 'C', 't0']:
        r, p = stats.pearsonr(sub['age'], sub[par])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"    {par}: r = {r:+.3f}, p = {p:.4f} {sig}")
print("\n  These six correlations are uncorrected. With Holm correction across the")
print("  family, none survives; treat the expert t0 x age result as exploratory.")

# --- LMM: age within each group ---
print("\n--- Age effect within each group (LMM) ---")
for grp in ['Novice', 'Expert']:
    sub = dat_age[dat_age['group'] == grp].copy()
    if sub['age'].notna().all():
        md = smf.mixedlm("P ~ age * font", sub, groups=sub["participant"]).fit(reml=True)
        age_coef = md.params.get('age', np.nan)
        age_p = md.pvalues.get('age', np.nan)
        print(f"  {grp}: age coef = {age_coef:.4f}, p = {age_p:.4f}")

## 7. H2 — Exposure-time differences between groups

### 7.1 Comparison at intended exposure times (Figure S8)

In [ ]:
# 7.1 H2: Expert vs Novice at each intended exposure time (Figure S8)
# ============================================================

print("=" * 75)
print("H2: Expert vs Novice at EACH Exposure Time (intended)")
print("=" * 75)

# Use intended times for common comparison
agg_time = (
    dat_age.groupby(['participant', 'group', 't_intended'])
    .agg(mean_P=('P', 'mean'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
results_per_t = []

for t_val in sorted(agg_time['t_intended'].unique()):
    sub = agg_time[agg_time['t_intended'] == t_val]
    nov = sub[sub['group'] == 'Novice']['mean_P']
    exp = sub[sub['group'] == 'Expert']['mean_P']
    if len(nov) > 1 and len(exp) > 1:
        t_s, p_v = ttest_ind(exp, nov)
        d = pg.compute_effsize(exp, nov, eftype='cohen')
    else:
        t_s, p_v, d = np.nan, np.nan, np.nan
    sig = '***' if p_v < 0.001 else '**' if p_v < 0.01 else '*' if p_v < 0.05 else 'ns'
    results_per_t.append({'t_intended': t_val, 'Nov_mean': nov.mean(), 'Exp_mean': exp.mean(),
                          'diff': exp.mean()-nov.mean(), 't_stat': t_s, 'p_value': p_v,
                          'Cohens_d': d, 'Sig': sig})

df_per_t = pd.DataFrame(results_per_t)
display(df_per_t.round(4))

# Plot group × time interaction
for grp, color, ls in [('Novice', '#1f77b4', '-'), ('Expert', '#d62728', '--')]:
    sub = agg_time[agg_time['group']==grp]
    means = sub.groupby('t_intended')['mean_P'].mean()
    sems = sub.groupby('t_intended')['mean_P'].sem()
    ax.errorbar(means.index, means.values, yerr=sems.values, marker='o',
               capsize=4, label=grp, color=color, ls=ls, lw=2)
    # Add sig markers
    for _, row in df_per_t.iterrows():
        if row['Sig'] != 'ns':
            t_x = row['t_intended']
            y_pos = max(means.get(t_x, 0) + sems.get(t_x, 0) + 0.15, 0)
            ax.text(t_x, y_pos + 0.1, row['Sig'], ha='center', fontsize=9, color='red')

ax.set_xlabel('Intended Exposure Time (ms)')
ax.set_ylabel('Mean Number Correct')
ax.set_title('H2: Expert vs Novice per Exposure Time')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
savefig("FigS08_group_by_intended_exposure")
plt.show()

### 7.2 Comparison at matched exposure-time pairs (Figure 8)

In [ ]:
# 7.2 H2: Expert vs Novice at matched exposure-time pairs (Figure 8)
# ============================================================
# Novice: uses corrected (frame-snapped) times, excluding 83.33 & 200 ms
# Expert: uses original intended times
# Comparison: at exposure times that are shared or near-shared between groups

print("=" * 75)
print("H2: Expert vs Novice at EACH Exposure Time")
print("    Novice → corrected times | Expert → original times")
print("=" * 75)

# --- A) Per-group means at their ACTUAL exposure times ---
agg_novice = (
    dat_age[dat_age['group'] == 'Novice']
    .groupby(['participant', 't'])          # 't' = corrected time
    .agg(mean_P=('P', 'mean'))
    .reset_index()
)
agg_expert = (
    dat_age[dat_age['group'] == 'Expert']
    .groupby(['participant', 't'])          # 't' = original intended time
    .agg(mean_P=('P', 'mean'))
    .reset_index()
)

novice_times = sorted(agg_novice['t'].unique())
expert_times = sorted(agg_expert['t'].unique())

print(f"\nNovice corrected times: {[f'{t:.2f}' for t in novice_times]} ms")
print(f"Expert original  times: {[f'{t:.0f}' for t in expert_times]} ms")

# --- B) Define comparable pairs (intended → corrected mapping) ---
# Each pair links the Expert intended time to its Novice corrected equivalent
COMPARE_PAIRS = [
    # (Expert time,  Novice corrected,  label)
    (10,    16.6667,   '10 → 16.67'),
    (20,    33.3333,   '20 → 33.33'),
    (30,    33.3333,   '30 → 33.33'),
    (40,    50.0000,   '40 → 50.00'),
    (50,    66.6667,   '50 → 66.67'),
    (60,    66.6667,   '60 → 66.67'),
    (90,   100.0000,   '90 → 100.0'),
    (200,  216.6667,  '200 → 216.7'),
]

# --- C) T-tests at each comparable pair ---
results_per_t = []
for t_exp, t_nov, label in COMPARE_PAIRS:
    nov = agg_novice[np.abs(agg_novice['t'] - t_nov) < 0.1]['mean_P']
    exp = agg_expert[np.abs(agg_expert['t'] - t_exp) < 0.1]['mean_P']

    if len(nov) > 1 and len(exp) > 1:
        t_s, p_v = ttest_ind(exp, nov)
        d = pg.compute_effsize(exp, nov, eftype='cohen')
    else:
        t_s, p_v, d = np.nan, np.nan, np.nan

    sig = '***' if p_v < 0.001 else '**' if p_v < 0.01 else '*' if p_v < 0.05 else 'ns'
    results_per_t.append({
        'Expert_t_ms': t_exp, 'Novice_t_ms': round(t_nov, 2),
        'Pair': label,
        'Nov_mean': nov.mean(), 'Exp_mean': exp.mean(),
        'Diff': exp.mean() - nov.mean(),
        't_stat': t_s, 'p_value': p_v, 'Cohens_d': d, 'Sig': sig
    })

df_per_t = pd.DataFrame(results_per_t)
# Holm correction across the eight duration comparisons
_, df_per_t['p_holm'], _, _ = multipletests(df_per_t['p_value'].values, alpha=0.05, method='holm')
print("\n=== Pairwise t-tests: Expert (original) vs Novice (corrected); p_holm across the 8 pairs ===")
display(df_per_t.round(4))
df_per_t.to_csv(TABLE_DIR / "matched_exposure_pairs.csv", index=False)

# --- D) Figure: Both groups on same axis, each at their own times ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1: Separate curves at actual times
ax = axes[0]
for grp, agg_grp, color, ls, marker in [
    ('Novice',  agg_novice, '#1f77b4', '-',  's'),
    ('Expert',  agg_expert, '#d62728', '--', 'o'),
]:
    means = agg_grp.groupby('t')['mean_P'].mean()
    sems  = agg_grp.groupby('t')['mean_P'].sem()
    ax.errorbar(means.index, means.values, yerr=sems.values, marker=marker,
                capsize=4, label=grp, color=color, ls=ls, lw=2, markersize=6)

ax.set_xlabel('Exposure Time (ms)\n(Novice: corrected | Expert: original)')
ax.set_ylabel('Mean Number Correct')
ax.set_title('Group Performance at Actual Exposure Times')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-0.3, 5.5)

# Panel 2: Paired comparison — plot both at midpoint of each pair
ax2 = axes[1]
x_pos = np.arange(len(COMPARE_PAIRS))
nov_means = df_per_t['Nov_mean'].values
exp_means = df_per_t['Exp_mean'].values
width = 0.35

bars_n = ax2.bar(x_pos - width/2, nov_means, width, label='Novice', color='#1f77b4', alpha=0.7)
bars_e = ax2.bar(x_pos + width/2, exp_means, width, label='Expert', color='#d62728', alpha=0.7)

# Add significance markers
for i, row in df_per_t.iterrows():
    if row['Sig'] != 'ns':
        y_top = max(row['Nov_mean'], row['Exp_mean']) + 0.2
        ax2.plot([i - width/2, i - width/2, i + width/2, i + width/2],
                 [y_top - 0.05, y_top, y_top, y_top - 0.05], 'k-', lw=0.8)
        ax2.text(i, y_top + 0.05, f"{row['Sig']}\np={row['p_value']:.3f}",
                 ha='center', fontsize=7, color='red')

ax2.set_xticks(x_pos)
ax2.set_xticklabels([p[2] for p in COMPARE_PAIRS], rotation=45, ha='right', fontsize=9)
ax2.set_xlabel('Exposure Time Pair (Expert → Novice corrected)')
ax2.set_ylabel('Mean Number Correct')
ax2.set_title('Paired Comparison: Expert vs Novice')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')
ax2.set_ylim(0, 5.8)

fig.suptitle('H2: Expert vs Novice per Exposure Time (Novice at corrected times)',
             fontsize=14, y=1.02)
fig.tight_layout()
savefig("Fig08_matched_exposure_pairs")
plt.show()

## 8. Dynamic time warping and age × performance detail (Figures S5–S7, Figure 12, Table 5)

In [ ]:
# 8. Dynamic time warping (Novice vs Expert exposure alignment) and age x performance detail
# ============================================================
# The Novice and Expert groups have DIFFERENT exposure time axes after
# correction (Novice: 6 frame-snapped times; Expert: 8 intended times).
# DTW finds the optimal non-linear alignment between these two
# performance curves, revealing which time points are functionally
# equivalent across groups.

from dtw import dtw as dtw_align
from scipy.stats import pearsonr, spearmanr, mannwhitneyu

print("=" * 80)
print("DYNAMIC TIME WARPING: Novice vs Expert Performance Curves")
print("=" * 80)

# ================================================================
# A) Empirical performance curves per Font × Group at actual times
# ================================================================

# Novice: corrected times; Expert: original times
emp_curves = {}
for grp in ['Novice', 'Expert']:
    for font in fonts_list:
        sub = dat_age[(dat_age['group']==grp) & (dat_age['font']==font)]
        curve = sub.groupby('t')['P'].mean().sort_index()
        emp_curves[(font, grp)] = curve

# Print the raw curves
for font in fonts_list:
    print(f"\n--- {font} ---")
    for grp in ['Novice', 'Expert']:
        c = emp_curves[(font, grp)]
        print(f"  {grp:>7s} times: {[f'{t:.1f}' for t in c.index]} ms")
        print(f"  {grp:>7s} P(t):  {[f'{v:.3f}' for v in c.values]}")

# ================================================================
# B) DTW alignment per font
# ================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

dtw_results = {}
for fi, font in enumerate(fonts_list):
    nov_curve = emp_curves[(font, 'Novice')]
    exp_curve = emp_curves[(font, 'Expert')]

    nov_t = nov_curve.index.values
    exp_t = exp_curve.index.values
    nov_p = nov_curve.values
    exp_p = exp_curve.values

    # Run DTW on the performance values
    alignment = dtw_align(nov_p, exp_p, keep_internals=True)
    dtw_results[font] = alignment

    print(f"\n{'='*60}")
    print(f"DTW alignment: {font}")
    print(f"  Normalized distance: {alignment.normalizedDistance:.4f}")
    print(f"  Warping path (Novice idx → Expert idx):")
    print(f"  {'Novice t (ms)':>15} {'Novice P':>10} {'Expert t (ms)':>15} {'Expert P':>10}")
    for i1, i2 in zip(alignment.index1, alignment.index2):
        print(f"  {nov_t[i1]:>13.2f}ms {nov_p[i1]:>10.3f} {exp_t[i2]:>13.0f}ms {exp_p[i2]:>10.3f}")

    # --- Panel 1 (top): Three-way DTW plot ---
    ax = axes[0, fi]

    # Plot both curves offset vertically for clarity
    offset = 0
    ax.plot(nov_t, nov_p + offset, 's-', color='#1f77b4', lw=2, markersize=8,
            label='Novice (corrected times)', zorder=3)
    ax.plot(exp_t, exp_p + offset, 'o--', color='#d62728', lw=2, markersize=8,
            label='Expert (original times)', zorder=3)

    # Draw warping lines connecting aligned points
    for i1, i2 in zip(alignment.index1, alignment.index2):
        ax.plot([nov_t[i1], exp_t[i2]],
                [nov_p[i1], exp_p[i2]],
                'k-', alpha=0.3, lw=0.8)

    ax.set_xlabel('Exposure time (ms)')
    ax.set_ylabel('Mean Number Correct')
    ax.set_title(f'{font}\nDTW dist = {alignment.normalizedDistance:.3f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.3, 5.8)

    # --- Panel 2 (bottom): Cost matrix + warping path ---
    ax2 = axes[1, fi]
    # Build cost matrix
    cost_matrix = np.zeros((len(nov_p), len(exp_p)))
    for i in range(len(nov_p)):
        for j in range(len(exp_p)):
            cost_matrix[i, j] = (nov_p[i] - exp_p[j])**2

    im = ax2.imshow(cost_matrix, origin='lower', cmap='YlOrRd', aspect='auto',
                     interpolation='nearest')
    # Overlay warping path
    ax2.plot(alignment.index2, alignment.index1, 'b-o', lw=2, markersize=5,
             label='Warping path')
    ax2.set_xticks(range(len(exp_t)))
    ax2.set_xticklabels([f'{t:.0f}' for t in exp_t], fontsize=8)
    ax2.set_yticks(range(len(nov_t)))
    ax2.set_yticklabels([f'{t:.1f}' for t in nov_t], fontsize=8)
    ax2.set_xlabel('Expert exposure time (ms)')
    ax2.set_ylabel('Novice corrected time (ms)')
    ax2.set_title(f'{font}: DTW Cost Matrix & Warping Path')
    ax2.legend(fontsize=8, loc='upper left')
    plt.colorbar(im, ax=ax2, shrink=0.8, label='Squared difference')

fig.suptitle('Dynamic Time Warping: Novice (corrected) vs Expert (original) Performance Curves',
             fontsize=14, y=1.02)
fig.tight_layout()
savefig("FigS05_dtw_alignment_empirical")
plt.show()

# ================================================================
# C) DTW on POSTERIOR TVA curves (population-level)
# ================================================================

print("\n" + "=" * 80)
print("DTW ON POSTERIOR TVA CURVES (median predictions) — DESCRIPTIVE ONLY")
print("=" * 80)

# Use the t_grid and tva_curve from Section 4.1
# Evaluate at fine resolution, then extract at empirical time points
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for fi, font in enumerate(fonts_list):
    ax = axes[fi]

    # Posterior median TVA curves evaluated at each group's actual times
    K_n, C_n, t0_n = get_KCt0(font, 'Novice')
    K_e, C_e, t0_e = get_KCt0(font, 'Expert')

    nov_t_actual = np.array(sorted(dat_age[(dat_age['group']=='Novice') &
                                            (dat_age['font']==font)]['t'].unique()))
    exp_t_actual = np.array(sorted(dat_age[(dat_age['group']=='Expert') &
                                            (dat_age['font']==font)]['t'].unique()))

    # TVA prediction at actual time points (posterior median)
    nov_t_s = nov_t_actual / 1000.0
    exp_t_s = exp_t_actual / 1000.0

    P_nov_all = tva_curve(K_n, C_n, t0_n, nov_t_s)  # (n_samples, n_nov_t)
    P_exp_all = tva_curve(K_e, C_e, t0_e, exp_t_s)  # (n_samples, n_exp_t)

    P_nov_med = np.median(P_nov_all, axis=0)
    P_exp_med = np.median(P_exp_all, axis=0)

    # DTW on posterior median curves
    alignment_post = dtw_align(P_nov_med, P_exp_med, keep_internals=True)

    print(f"\n  {font}: DTW distance (posterior) = {alignment_post.normalizedDistance:.4f}")
    print(f"  Warping path:")
    for i1, i2 in zip(alignment_post.index1, alignment_post.index2):
        print(f"    Novice {nov_t_actual[i1]:>7.2f}ms (P={P_nov_med[i1]:.3f})"
              f"  ↔  Expert {exp_t_actual[i2]:>5.0f}ms (P={P_exp_med[i2]:.3f})")

    # Plot
    ax.plot(nov_t_actual, P_nov_med, 's-', color='#1f77b4', lw=2, markersize=8,
            label='Novice (posterior median)')
    ax.plot(exp_t_actual, P_exp_med, 'o--', color='#d62728', lw=2, markersize=8,
            label='Expert (posterior median)')

    for i1, i2 in zip(alignment_post.index1, alignment_post.index2):
        ax.annotate('', xy=(exp_t_actual[i2], P_exp_med[i2]),
                    xytext=(nov_t_actual[i1], P_nov_med[i1]),
                    arrowprops=dict(arrowstyle='->', color='grey', alpha=0.5, lw=1.2))

    ax.set_xlabel('Exposure time (ms)')
    ax.set_ylabel('Predicted Number Correct (median)')
    ax.set_title(f'{font}\nDTW dist = {alignment_post.normalizedDistance:.3f}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('DTW on Posterior TVA Curves', fontsize=14, y=1.02)
fig.tight_layout()
savefig("FigS06_dtw_posterior_curves")
plt.show()

# ================================================================
# D) AGE × PERFORMANCE: Detailed statistical analysis
# ================================================================

print("\n" + "=" * 80)
print("AGE × PERFORMANCE ANALYSIS")
print("=" * 80)

# --- D1: Is there a significant age difference between groups? ---
nov_ages = participant_info[participant_info['group'] == 'Novice']['age'].values
exp_ages = participant_info[participant_info['group'] == 'Expert']['age'].values

t_age, p_age = ttest_ind(nov_ages, exp_ages)
u_age, p_u_age = mannwhitneyu(nov_ages, exp_ages, alternative='two-sided')
d_age = pg.compute_effsize(nov_ages, exp_ages, eftype='cohen')

print(f"\n--- D1: Age difference between groups ---")
print(f"  Novice: M = {nov_ages.mean():.1f}, SD = {nov_ages.std():.1f}, "
      f"range = {nov_ages.min()}–{nov_ages.max()}")
print(f"  Expert: M = {exp_ages.mean():.1f}, SD = {exp_ages.std():.1f}, "
      f"range = {exp_ages.min()}–{exp_ages.max()}")
print(f"  Independent t-test: t({len(nov_ages)+len(exp_ages)-2}) = {t_age:.3f}, p = {p_age:.4f}")
print(f"  Mann-Whitney U: U = {u_age:.1f}, p = {p_u_age:.4f}")
print(f"  Cohen's d: {d_age:.3f}")
print(f"  → {'Significant' if p_age < 0.05 else 'No significant'} age difference between groups")

# --- D2: Performance vs Age — within each group ---
print(f"\n--- D2: Age–Performance correlations within each group ---")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Top row: raw performance vs age
for gi, grp in enumerate(['Novice', 'Expert']):
    for fi, font in enumerate(fonts_list):
        sub = df_part_font[(df_part_font['group']==grp) & (df_part_font['font']==font)]
        r, p = pearsonr(sub['age'], sub['mean_P'])
        rho, p_rho = spearmanr(sub['age'], sub['mean_P'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"  {grp} × {font[:8]:8s}: Pearson r = {r:+.3f} (p={p:.4f} {sig}), "
              f"Spearman ρ = {rho:+.3f} (p={p_rho:.4f})")

# Top row: K vs Age scatter with both groups and regression
param_labels = {'K': 'K (VSTM Capacity)', 'C': 'C (Processing Speed)',
                't0': 't₀ (Perceptual Threshold)'}

for pi, par in enumerate(['K', 'C', 't0']):
    ax = axes[0, pi]
    for grp, color, marker in [('Novice', '#1f77b4', 's'), ('Expert', '#d62728', 'o')]:
        sub = df_kct[df_kct['group']==grp].groupby('participant').agg(
            {par: 'mean', 'age': 'first'}).reset_index()

        ax.scatter(sub['age'], sub[par], c=color, marker=marker, s=60, alpha=0.7,
                  label=grp, edgecolors='black', linewidth=0.3)

        # Regression line
        z = np.polyfit(sub['age'], sub[par], 1)
        poly = np.poly1d(z)
        x_line = np.linspace(sub['age'].min(), sub['age'].max(), 50)
        ax.plot(x_line, poly(x_line), color=color, ls='--', lw=1.5, alpha=0.7)

        r, p = pearsonr(sub['age'], sub[par])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_text = 0.95 if grp == 'Novice' else 0.85
        ax.text(0.03, y_text, f'{grp}: r={r:+.3f} (p={p:.3f}) {sig}',
                transform=ax.transAxes, fontsize=8, color=color,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_xlabel('Age (years)')
    ax.set_ylabel(param_labels[par])
    ax.set_title(f'{par} vs Age', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Bottom row: TVA parameters across age bands by group

# Split each group into age tertiles
for pi, par in enumerate(['K', 'C', 't0']):
    ax = axes[1, pi]

    age_bins = [20, 35, 50, 70]
    age_labels = ['Young\n(≤35)', 'Middle\n(36–50)', 'Senior\n(>50)']

    box_data = []
    box_labels = []
    box_colors = []
    positions = []

    for ai, a_label in enumerate(age_labels):
        lo, hi = age_bins[ai], age_bins[ai + 1]
        for gi, (grp, color) in enumerate([('Novice', '#1f77b4'), ('Expert', '#d62728')]):
            sub = df_kct[df_kct['group']==grp].groupby('participant').agg(
                {par: 'mean', 'age': 'first'}).reset_index()
            sub_age = sub[(sub['age'] > lo) & (sub['age'] <= hi)]

            pos = ai * 3 + gi
            if len(sub_age) > 0:
                bp = ax.boxplot([sub_age[par]], positions=[pos], widths=0.8,
                               patch_artist=True, showmeans=True,
                               meanprops=dict(marker='D', markerfacecolor='white', markersize=5))
                bp['boxes'][0].set_facecolor(color)
                bp['boxes'][0].set_alpha(0.5)
                ax.scatter(np.random.normal(pos, 0.08, len(sub_age)), sub_age[par],
                          color=color, alpha=0.6, s=25, zorder=3)
                ax.text(pos, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else sub_age[par].min()*0.9,
                        f'n={len(sub_age)}', ha='center', fontsize=7, va='top')

    ax.set_xticks([0.5, 3.5, 6.5])
    ax.set_xticklabels(age_labels)
    ax.set_ylabel(param_labels[par])
    ax.set_title(f'{par}: Age Group × Expertise', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.2, axis='y')

# Add legend to last panel
from matplotlib.patches import Patch
axes[1, 2].legend(handles=[
    Patch(facecolor='#1f77b4', alpha=0.5, label='Novice'),
    Patch(facecolor='#d62728', alpha=0.5, label='Expert')
], loc='best', fontsize=9)

fig.suptitle('Age × Expertise: TVA parameters across age bands (descriptive; no interaction detected)', fontsize=14, y=1.02)
fig.tight_layout()
savefig("FigS07_age_bands_tva_parameters")
plt.show()

# --- D3: Descriptive comparison — oldest experts vs youngest novices (n=3 per subgroup; illustrative only) ---
print(f"\n--- D3: Oldest Experts vs Youngest Novices (descriptive, n=3 per subgroup) ---")
print("Descriptive comparison only — n = 3 per subgroup, no inference supported.\n")

df_kct_agg = df_kct.groupby(['participant', 'group']).agg(
    {'K': 'mean', 'C': 'mean', 't0': 'mean', 'age': 'first'}).reset_index()

oldest_experts = df_kct_agg[df_kct_agg['group']=='Expert'].nlargest(3, 'age')
youngest_novices = df_kct_agg[df_kct_agg['group']=='Novice'].nsmallest(3, 'age')

print("  3 Oldest Experts:")
for _, r in oldest_experts.iterrows():
    print(f"    {r['participant']:>15s}  age={int(r['age'])}  K={r['K']:.3f}  "
          f"C={r['C']:.1f}  t0={r['t0']*1000:.1f}ms")

print("\n  3 Youngest Novices:")
for _, r in youngest_novices.iterrows():
    print(f"    {r['participant']:>15s}  age={int(r['age'])}  K={r['K']:.3f}  "
          f"C={r['C']:.1f}  t0={r['t0']*1000:.1f}ms")

# Compare means
for par in ['K', 'C', 't0']:
    oe_mean = oldest_experts[par].mean()
    yn_mean = youngest_novices[par].mean()
    advantage = 'Expert' if (par in ['K','C'] and oe_mean > yn_mean) or \
                            (par == 't0' and oe_mean < yn_mean) else 'Novice'
    print(f"\n  {par}: Oldest Expert mean = {oe_mean:.3f}, Youngest Novice mean = {yn_mean:.3f}"
          f"  → {advantage} advantage")

table5 = pd.DataFrame({
    'Subgroup': ['3 oldest Experts', '3 youngest Novices'],
    'ages': [', '.join(str(int(a)) for a in sorted(oldest_experts['age'])),
             ', '.join(str(int(a)) for a in sorted(youngest_novices['age']))],
    'mean_age': [oldest_experts['age'].mean(), youngest_novices['age'].mean()],
    'K_letters': [oldest_experts['K'].mean(), youngest_novices['K'].mean()],
    'C_letters_per_s': [oldest_experts['C'].mean(), youngest_novices['C'].mean()],
    't0_ms': [oldest_experts['t0'].mean() * 1000, youngest_novices['t0'].mean() * 1000],
})
table5.to_csv(TABLE_DIR / "Table5_age_extremes.csv", index=False)

# --- D4: ANCOVA-style LMM — does group effect survive controlling for age? ---
print(f"\n--- D4: LMM controlling for age ---")
print("  Does the Expert advantage remain after accounting for age?\n")

# Trial-level LMM: P ~ group + age + group:age + (1|participant)
lmm_age = smf.mixedlm(
    "P ~ group + age + group:age",
    dat_age, groups=dat_age["participant"]
).fit(reml=True)

print("  Model: P ~ group + age + group:age + (1|participant)")
print(f"  Group effect:       coef = {lmm_age.params.get('group[T.Novice]', np.nan):.4f}, "
      f"p = {lmm_age.pvalues.get('group[T.Novice]', np.nan):.4f}")
print(f"  Age effect:         coef = {lmm_age.params.get('age', np.nan):.4f}, "
      f"p = {lmm_age.pvalues.get('age', np.nan):.4f}")

# Check for interaction
interaction_keys = [k for k in lmm_age.params.keys() if 'group' in k.lower() and 'age' in k.lower()]
if interaction_keys:
    k = interaction_keys[0]
    print(f"  Group × Age:        coef = {lmm_age.params[k]:.4f}, p = {lmm_age.pvalues[k]:.4f}")

print(f"\n  → {'Group effect SURVIVES' if lmm_age.pvalues.get('group[T.Novice]', 1) < 0.05 else 'Group effect NOT significant'} after controlling for age")
print(f"  → {'Age IS' if lmm_age.pvalues.get('age', 1) < 0.05 else 'Age is NOT'} a significant predictor of performance")

# --- D5: Summary figure — paired age vs K scatter with arrows ---
fig, ax = plt.subplots(figsize=(10, 7))

for grp, color, marker in [('Novice', '#1f77b4', 's'), ('Expert', '#d62728', 'o')]:
    sub = df_kct_agg[df_kct_agg['group']==grp]
    ax.scatter(sub['age'], sub['K'], c=color, marker=marker, s=80, alpha=0.7,
              label=grp, edgecolors='black', linewidth=0.5, zorder=3)

    z = np.polyfit(sub['age'], sub['K'], 1)
    poly = np.poly1d(z)
    x_line = np.linspace(20, 68, 100)
    ax.plot(x_line, poly(x_line), color=color, ls='--', lw=2, alpha=0.5)

    r, p = pearsonr(sub['age'], sub['K'])
    slope = z[0]
    ax.text(0.03 if grp == 'Novice' else 0.60,
            0.97 if grp == 'Novice' else 0.97,
            f'{grp}:\n  r = {r:+.3f}, p = {p:.3f}\n  slope = {slope:+.4f} K/year',
            transform=ax.transAxes, fontsize=10, va='top', color=color,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=color))

# Highlight oldest expert and youngest novice
for _, r in oldest_experts.head(1).iterrows():
    ax.annotate(f"Oldest Expert\nage={int(r['age'])}, K={r['K']:.2f}",
                xy=(r['age'], r['K']), xytext=(r['age']-8, r['K']+0.3),
                fontsize=8, arrowprops=dict(arrowstyle='->', color='#d62728'),
                color='#d62728', fontweight='bold')
for _, r in youngest_novices.head(1).iterrows():
    ax.annotate(f"Youngest Novice\nage={int(r['age'])}, K={r['K']:.2f}",
                xy=(r['age'], r['K']), xytext=(r['age']+3, r['K']-0.4),
                fontsize=8, arrowprops=dict(arrowstyle='->', color='#1f77b4'),
                color='#1f77b4', fontweight='bold')

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('K (VSTM Capacity, mean over fonts)', fontsize=12)
ax.set_title('K (VSTM capacity) vs Age by Group\nNo evidence of a differential age-related decline ($p_{group \\times age}$ = .75)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
savefig("Fig12_capacity_vs_age")
plt.show()


## 9. Supplementary and sensitivity analyses

These sections do not replace anything above; they quantify the robustness checks referred to in the manuscript.

| Section | What it does |
|---------|--------------|
| **9.1** | Reconciles 5,760 → 5,440 → 5,281 trials exactly; audits per-participant retention (flags any participant who lost most of their trials to the frame-snap filter); quantifies the exposure-timing fault and shows that it is confounded with Group; shows that the "matched pairs" are only approximately matched. |
| **9.2** | Ceiling diagnostics for Helvetica: empirical rate of perfect reports, posterior mass of *K* near the *S* = 5 bound, credible-interval widths, and a ROPE check. The Helvetica null is **insufficient sensitivity**, not equivalence. |
| **9.3** | Fits TVA **independently** per participant × font (no partial pooling), measures how much shrinkage compressed the between-participant SD (Figure 4), and reruns the frequentist tests on the unshrunk estimates. |
| **9.4** | Refits the full hierarchical model with **Expert #8 reinstated** and prints the posterior contrasts side by side with the main analysis. |
| **9.5** | Bootstrap CI for the DTW distance, a within-group split-half noise floor, the scale explanation for Helvetica > Klimax Plus, and a direct estimate of the **horizontal (temporal) lead** (Figure 10). |
| **9.6** | Sensitivity power analysis, trials per cell, group composition, and the stimulus/display parameters that can be recovered from the data files. |
| **9.7** | Reconciles the parameter estimands (population-level posterior vs. participant-level averages) and produces the manuscript's parameter table (Table 3) from the independent fits. |

**Reporting rules**

1. Posterior contrasts are the **primary** group inference. Frequentist tests on model-derived parameters are secondary and multiplicity-corrected.
2. No p-values are computed **across posterior draws**. Draws express uncertainty about parameters, not independent observations.
3. Null results in the Helvetica condition are described as *not sensitive enough to resolve a difference*, never as evidence of equivalence.
4. The DTW analysis is descriptive. Temporal-lead claims rest on Section 9.5, not on the DTW distance.


### 9.1 Trial accounting, per-participant retention and the timing confound

In [ ]:
# 9.1 Trial accounting, per-participant retention, and the timing confound
# ============================================================================
# Reconciles the trial counts (5,760 recorded -> 5,440 after participant
# exclusions -> 5,281 analysed), audits per-participant retention, and
# quantifies the exposure-timing fault, which is confounded with Group.

print("=" * 78)
print("S1a. TRIAL ACCOUNTING — full reconciliation")
print("=" * 78)

n_raw          = len(dat_raw)
n_per_part     = 160
n_participants_raw = dat_raw['Participants_ID'].nunique()
n_excluded_part = 2
n_after_excl   = n_raw - n_excluded_part * n_per_part
n_final        = len(dat)
n_dropped_snap = n_after_excl - n_final

print(f"  Recorded trials                                   : {n_raw:>6d}"
      f"   ({n_participants_raw} participants x {n_per_part})")
print(f"  - R17_Novice (mean timing error ~100 ms)          : {-n_per_part:>6d}")
print(f"  - R8_Expert  (removed for age balance; see 9.4)   : {-n_per_part:>6d}")
print(f"  = Trials from the 34 analysed participants        : {n_after_excl:>6d}")
print(f"  - Novice trials whose frame-snapped duration did")
print(f"    not match the dominant mapping for their")
print(f"    intended duration                               : {-n_dropped_snap:>6d}"
      f"   ({100*n_dropped_snap/n_after_excl:.1f}%)")
print(f"  = Trials entering the TVA model                   : {n_final:>6d}")
assert n_final == n_after_excl - n_dropped_snap, "Trial accounting does not reconcile!"
print("\n  -> Reconciles exactly.")

# Per-group split
print("\n  By group:")
for grp in ['Novice', 'Expert']:
    n_g = (dat['group'] == grp).sum()
    n_g_expected = 17 * n_per_part
    print(f"    {grp:>6s}: {n_g:>5d} / {n_g_expected} retained "
          f"({100*n_g/n_g_expected:.1f}%)  — {n_g_expected - n_g} dropped")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S1b. PER-PARTICIPANT RETENTION — is the loss concentrated?")
print("=" * 78)

retained = dat.groupby(['participant', 'group']).size().rename('n_retained').reset_index()
retained['n_recorded'] = n_per_part
retained['pct_retained'] = 100 * retained['n_retained'] / retained['n_recorded']
retained = retained.sort_values('n_retained')

print("\n  Participants retaining < 95% of their trials:")
low = retained[retained['pct_retained'] < 95]
if len(low) == 0:
    print("    (none)")
else:
    print(low.to_string(index=False, float_format='{:.1f}'.format))

print(f"\n  Retention: min = {retained['n_retained'].min()} trials, "
      f"median = {retained['n_retained'].median():.0f}, "
      f"max = {retained['n_retained'].max()}")

n_severe = int((retained['pct_retained'] < 50).sum())
if n_severe:
    print(f"\n  ** WARNING: {n_severe} participant(s) retain < 50% of their trials. **")
    print("     Their frame-snapped durations largely failed to match the dominant")
    print("     mapping, i.e. their display timing was unstable in the same way as")
    print("     R17 but below the exclusion threshold. This must be reported, and")
    print("     a leave-one-out check run (below).")
    for _, r in retained[retained['pct_retained'] < 50].iterrows():
        print(f"       {r['participant']} ({r['group']}): "
              f"{r['n_retained']}/{n_per_part} trials "
              f"({r['pct_retained']:.0f}%)")

# Sensitivity: do the group cell means move if low-retention participants go?
if n_severe:
    drop_ids = retained[retained['pct_retained'] < 50]['participant'].tolist()
    print(f"\n  Cell means with vs. without {drop_ids}:")
    for font in fonts_list:
        for grp in ['Novice', 'Expert']:
            full = dat[(dat['font'] == font) & (dat['group'] == grp)]['P'].mean()
            trim = dat[(dat['font'] == font) & (dat['group'] == grp) &
                       (~dat['participant'].isin(drop_ids))]['P'].mean()
            print(f"    {font:>15s} {grp:>6s}: full = {full:.3f}, "
                  f"excl. = {trim:.3f}, delta = {trim - full:+.3f}")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S1c. TIMING FAULT — magnitude, and its confound with Group")
print("=" * 78)
print("  The timing error appears in exactly one group. Because Group is the")
print("  between-subjects factor, any hardware / room / display difference")
print("  between the two testing contexts is perfectly confounded with it.")
print("  It is quantified explicitly below.\n")

FRAME_MS = 1000.0 / 60.0
tim = dat_raw.copy()
tim['delta_t'] = tim['TargetDisplay_chrono'] - tim['TargetDisplay_ms']

print(f"  {'Group':>7s} {'n':>6s} {'mean dt':>9s} {'sd':>7s} {'median':>7s} "
      f"{'max':>6s} {'>1 frame':>10s} {'>3 frames':>10s}")
for grp in ['Novice', 'Expert']:
    s = tim[tim['GroupCondition'] == grp]['delta_t']
    print(f"  {grp:>7s} {len(s):>6d} {s.mean():>9.2f} {s.std():>7.2f} "
          f"{s.median():>7.1f} {s.max():>6.0f} "
          f"{100*(s.abs() > FRAME_MS).mean():>9.1f}% "
          f"{100*(s.abs() > 3*FRAME_MS).mean():>9.1f}%")

# Residual (uncorrected) trial-level scatter AFTER the dominant-only frame snap
print("\n  Residual trial-level timing scatter in the ANALYSED novice data")
print("  (frame-snap fixes the central tendency; it does not remove within-cell")
print("   variance, because each retained trial keeps its own true duration):")
nov_an = dat[dat['group'] == 'Novice'].copy()
nov_an['true_dur'] = nov_an['TargetDisplay_chrono']
print(f"\n  {'intended':>9s} {'snapped':>9s} {'n':>5s} {'true mean':>10s} "
      f"{'true sd':>8s} {'true min':>9s} {'true max':>9s}")
for t_int in sorted(nov_an['t_intended'].unique()):
    s = nov_an[nov_an['t_intended'] == t_int]
    print(f"  {t_int:>8.0f}  {s['t'].iloc[0]:>8.2f} {len(s):>5d} "
          f"{s['true_dur'].mean():>10.2f} {s['true_dur'].std():>8.2f} "
          f"{s['true_dur'].min():>9.0f} {s['true_dur'].max():>9.0f}")

print("\n  Expert durations for reference (chrono vs intended):")
exp_an = dat[dat['group'] == 'Expert']
for t_int in sorted(exp_an['t_intended'].unique()):
    s = exp_an[exp_an['t_intended'] == t_int]
    print(f"  {t_int:>8.0f}  {t_int:>8.2f} {len(s):>5d} "
          f"{s['TargetDisplay_chrono'].mean():>10.2f} "
          f"{s['TargetDisplay_chrono'].std():>8.2f} "
          f"{s['TargetDisplay_chrono'].min():>9.0f} "
          f"{s['TargetDisplay_chrono'].max():>9.0f}")

print("\n  -> 'Matched pairs' such as Expert 200 ms vs Novice 216.67 ms are NOT")
print("     matched: the novice cell is ~8% longer, and its within-cell spread is")
print("     larger. Every duration-wise group comparison inherits this bias, and")
print("     the bias favours the NOVICE group (longer true exposures), so it")
print("     works against the expertise hypothesis rather than for it.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S1d. TIMING-ROBUST CHECK — model novice trials at their TRUE durations")
print("=" * 78)
print("  The frame-snap assigns every trial in a cell the dominant duration.")
print("  As a robustness check, refit the descriptive curves using each novice")
print("  trial's own recorded chrono duration instead of the snapped value.\n")

nov_true = dat[dat['group'] == 'Novice'].copy()
nov_true['t_true_ms'] = nov_true['TargetDisplay_chrono'].astype(float)
by_true = (nov_true.groupby(['font', 't_true_ms'])
           .agg(mean_P=('P', 'mean'), n=('P', 'size'))
           .reset_index())
print("  Novice mean P at each distinct TRUE duration (n >= 20 only):")
print(by_true[by_true['n'] >= 20].to_string(index=False, float_format='{:.3f}'.format))
print("\n  Compare with the snapped-duration curves used in the main analysis;")
print("  large divergence would indicate the frame-snap is doing real work.")

### 9.2 Ceiling and sensitivity diagnostics for the Helvetica condition (ROPE)

In [ ]:
# 9.2 Ceiling / sensitivity diagnostics for the Helvetica condition
# ============================================================================
# K is bounded above by the display set size S = 5. In this model K is
# parameterised as K = 5 * sigmoid(klin), so K < 5 BY CONSTRUCTION and the
# likelihood carries almost no information about how far above ~4.5 a
# participant's true capacity lies. If both groups' true K approach or exceed
# 5 in the easy font, the design cannot detect a difference between them.
# The quantities below establish how close the Helvetica condition sits to
# that bound, so the null can be reported as INSUFFICIENT SENSITIVITY rather
# than as evidence of equivalence.

print("=" * 78)
print("S2a. EMPIRICAL CEILING — how often is the display exhausted?")
print("=" * 78)

S_SET = 5
cei = dat.copy()
cei['perfect'] = (cei['P'] >= S_SET).astype(int)
cei['near_ceiling'] = (cei['P'] >= S_SET - 1).astype(int)

print(f"\n  {'Font':>15s} {'Group':>7s} {'n':>6s} {'mean P':>8s} "
      f"{'% P=5':>8s} {'% P>=4':>8s}")
for font in fonts_list:
    for grp in ['Novice', 'Expert']:
        s = cei[(cei['font'] == font) & (cei['group'] == grp)]
        print(f"  {font:>15s} {grp:>7s} {len(s):>6d} {s['P'].mean():>8.3f} "
              f"{100*s['perfect'].mean():>7.1f}% {100*s['near_ceiling'].mean():>7.1f}%")

print("\n  At the LONGEST exposure only (the asymptote that identifies K):")
print(f"  {'Font':>15s} {'Group':>7s} {'t (ms)':>8s} {'n':>5s} {'mean P':>8s} "
      f"{'% P=5':>8s} {'% P>=4':>8s}")
for font in fonts_list:
    for grp in ['Novice', 'Expert']:
        sub = cei[(cei['font'] == font) & (cei['group'] == grp)]
        t_max = sub['t'].max()
        s = sub[sub['t'] == t_max]
        print(f"  {font:>15s} {grp:>7s} {t_max:>8.2f} {len(s):>5d} "
              f"{s['P'].mean():>8.3f} {100*s['perfect'].mean():>7.1f}% "
              f"{100*s['near_ceiling'].mean():>7.1f}%")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S2b. POSTERIOR PROXIMITY OF K TO THE S = 5 BOUND")
print("=" * 78)
print("  P(K > 4.5) and P(K > 4.75) index how much of the posterior is pressed")
print("  against the ceiling. Values that are large for Helvetica but small for")
print("  Klimax Plus show the easy font is the compressed condition.\n")

print(f"  {'Font':>15s} {'Group':>7s} {'mean K':>8s} {'95% CrI':>20s} "
      f"{'P(K>4.5)':>9s} {'P(K>4.75)':>10s}")
for font in fonts_list:
    for grp in groups_list:
        K, C, t0 = get_KCt0(font, grp)
        lo, hi = np.quantile(K, [0.025, 0.975])
        print(f"  {font:>15s} {grp:>7s} {K.mean():>8.3f} "
              f"[{lo:>6.3f}, {hi:>6.3f}]   "
              f"{(K > 4.5).mean():>9.3f} {(K > 4.75).mean():>10.3f}")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S2c. WHAT DIFFERENCE COULD THE HELVETICA CELL HAVE DETECTED?")
print("=" * 78)
print("  Width of the posterior on the Helvetica group contrast, expressed as")
print("  the largest true difference that is still comfortably inside the CrI.")
print("  A wide interval means 'uninformative', not 'no difference'.\n")

for par_name in ['K', 'C', 't0']:
    for font in fonts_list:
        Ke, Ce, t0e = get_KCt0(font, 'Expert')
        Kn, Cn, t0n = get_KCt0(font, 'Novice')
        d = {'K': Ke - Kn, 'C': Ce - Cn, 't0': (t0e - t0n) * 1000}[par_name]
        lo, hi = np.quantile(d, [0.025, 0.975])
        unit = {'K': 'items', 'C': 'items/s', 't0': 'ms'}[par_name]
        print(f"  {par_name:>2s}  {font:>15s}: mean {d.mean():+7.3f}, "
              f"95% CrI [{lo:+7.3f}, {hi:+7.3f}] {unit}"
              f"   (interval width {hi-lo:.3f})")
    print()

print("  ROPE check — is the Helvetica contrast inside a region of practical")
print("  equivalence? Only if it is can 'no difference' be asserted.")
ROPE = {'K': 0.25, 'C': 10.0, 't0': 2.0}   # items, items/s, ms  (declare & justify)
for par_name, rope in ROPE.items():
    for font in fonts_list:
        Ke, Ce, t0e = get_KCt0(font, 'Expert')
        Kn, Cn, t0n = get_KCt0(font, 'Novice')
        d = {'K': Ke - Kn, 'C': Ce - Cn, 't0': (t0e - t0n) * 1000}[par_name]
        in_rope = np.mean(np.abs(d) < rope)
        print(f"  {par_name:>2s}  {font:>15s}: P(|diff| < {rope} ) = {in_rope:.3f}"
              f"  -> {'equivalence supported' if in_rope > 0.95 else 'INCONCLUSIVE'}")
    print()

print("  -> If the Helvetica contrasts are neither credibly non-zero nor inside")
print("     the ROPE, the correct wording is 'the Helvetica condition was not")
print("     sufficiently sensitive to resolve a group difference', NOT 'the")
print("     groups performed equivalently'.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S2d. WHY THE BOUND MATTERS — K is capped at S = 5 by construction")
print("=" * 78)
print(f"  Model: K = S * sigmoid(klin), S = {S_SET}. K can never exceed {S_SET}.")
print("  A participant whose true capacity is 5.5 items and one whose true")
print("  capacity is 7 items produce indistinguishable data with a 5-letter")
print("  display, and both are estimated at K -> 5. Any group difference in")
print("  true capacity above the display size is therefore unobservable here.")
print("  Larger set sizes (e.g. 6-8 letters) would be needed to resolve it.")

### 9.3 Independent per-participant TVA fits and shrinkage check (Figure 4)

In [ ]:
# 9.3 Independent per-participant TVA fits (no partial pooling) and shrinkage check
# ============================================================================
# The tests in Section 5.2 use per-participant K/C/t0 reconstructed
# from a hierarchical model that ALREADY contains Font x Group fixed effects.
# Partial pooling drags every participant toward their group x font mean, which
# (a) shrinks within-group variance and (b) imports the very group difference
# being tested. Both inflate Cohen's d and deflate p.
#
# Here each participant x font cell is fitted INDEPENDENTLY by non-linear least
# squares on that cell's trial data, with no information shared across
# participants or groups. The frequentist Font x Group tests are then rerun on
# these unshrunk estimates. Divergence between the two sets of estimates is a
# direct measure of how much the shrinkage was doing.

from scipy.optimize import least_squares
from statsmodels.stats.multitest import multipletests

S_SET = 5.0
K_BOUND = 5.0          # K cannot exceed the display set size
C_BOUND = 1000.0
T0_BOUND = 0.060       # 60 ms


def tva_pred(theta, t_s):
    """TVA exponential: P(t) = K * (1 - exp(-(C/K) * max(t - t0, 0)))."""
    K, C, t0 = theta
    dt = np.clip(t_s - t0, 0.0, None)
    return K * (1.0 - np.exp(-np.clip((C / max(K, 1e-9)) * dt, 0, 700)))


def fit_one_cell(t_s, P, n_starts=8, seed=0):
    """Independent least-squares TVA fit for one participant x font cell."""
    rng_l = np.random.default_rng(seed)
    lb = np.array([0.05, 1.0, 0.0])
    ub = np.array([K_BOUND, C_BOUND, T0_BOUND])

    starts = [np.array([min(P.max() if len(P) else 3.0, 4.5), 40.0, 0.020])]
    for _ in range(n_starts - 1):
        starts.append(np.array([rng_l.uniform(1.0, 4.8),
                                rng_l.uniform(10.0, 200.0),
                                rng_l.uniform(0.0, 0.040)]))

    best, best_cost = None, np.inf
    for x0 in starts:
        x0 = np.clip(x0, lb + 1e-9, ub - 1e-9)
        try:
            res = least_squares(lambda th: tva_pred(th, t_s) - P,
                                x0, bounds=(lb, ub), max_nfev=4000)
        except Exception:
            continue
        if res.cost < best_cost:
            best, best_cost = res, res.cost

    if best is None:
        return dict(K=np.nan, C=np.nan, t0=np.nan, rss=np.nan,
                    converged=False, K_at_bound=False)

    K, C, t0 = best.x
    return dict(K=K, C=C, t0=t0, rss=2 * best.cost,
                converged=bool(best.success),
                K_at_bound=bool(K > K_BOUND - 1e-3))


print("=" * 78)
print("S3a. Fitting TVA independently for each participant x font cell")
print("=" * 78)

rows_indep = []
# Deterministic seeds (Python's string hash is salted per process, so hash()
# would make the multi-start fits irreproducible across runs).
for _cell_i, ((pname, grp, font), sub) in enumerate(
        dat.groupby(['participant', 'group', 'font'], sort=True)):
    t_s_cell = sub['t_s'].to_numpy(dtype=float)
    P_cell = sub['P'].to_numpy(dtype=float)
    fit = fit_one_cell(t_s_cell, P_cell, seed=1000 + _cell_i)
    age_row = participant_info[participant_info['Participants_ID'] == pname]
    rows_indep.append({
        'participant': pname, 'group': grp, 'font': font,
        'n_trials': len(sub), 'n_times': sub['t'].nunique(),
        'age': int(age_row['age'].iloc[0]) if len(age_row) else np.nan,
        **fit
    })

df_indep = pd.DataFrame(rows_indep)
df_indep.to_csv(TABLE_DIR / "participant_parameters_independent_fits.csv", index=False)
n_bad = int((~df_indep['converged']).sum())
n_bound = int(df_indep['K_at_bound'].sum())
print(f"  Cells fitted      : {len(df_indep)}")
print(f"  Failed to converge: {n_bad}")
print(f"  K pinned at the S = 5 bound: {n_bound}  "
      f"({100*n_bound/len(df_indep):.1f}%)  <- direct ceiling evidence")
print("\n  K at the bound, by cell:")
print(df_indep.groupby(['font', 'group'])['K_at_bound'].agg(['sum', 'size']).to_string())

print("\n  Independent (unshrunk) parameter means:")
print(df_indep.groupby(['font', 'group'])[['K', 'C', 't0']]
      .agg(['mean', 'std']).round(3).to_string())

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S3b. HOW MUCH DID SHRINKAGE MATTER?")
print("=" * 78)

cmp = df_indep.merge(df_kct, on=['participant', 'group', 'font'],
                     suffixes=('_indep', '_hier'))
print(f"\n  {'Font':>15s} {'Par':>3s} {'SD indep':>9s} {'SD hier':>9s} "
      f"{'SD ratio':>9s} {'r':>7s}")
for font in fonts_list:
    for par in ['K', 'C', 't0']:
        s = cmp[cmp['font'] == font]
        sd_i = s[f'{par}_indep'].std()
        sd_h = s[f'{par}_hier'].std()
        r = np.corrcoef(s[f'{par}_indep'], s[f'{par}_hier'])[0, 1]
        print(f"  {font:>15s} {par:>3s} {sd_i:>9.3f} {sd_h:>9.3f} "
              f"{sd_h/sd_i:>9.3f} {r:>7.3f}")
print("\n  SD ratio < 1 means the hierarchical estimates are less variable than")
print("  the independent ones — that is the shrinkage. Any t-test built on the")
print("  shrunk values therefore has an artificially small denominator.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S3c. FREQUENTIST TESTS ON THE UNSHRUNK ESTIMATES (primary frequentist)")
print("=" * 78)

res_indep = []
for font in fonts_list:
    for par in ['K', 'C', 't0']:
        nov = df_indep[(df_indep['font'] == font) &
                       (df_indep['group'] == 'Novice')][par].dropna().values
        exp = df_indep[(df_indep['font'] == font) &
                       (df_indep['group'] == 'Expert')][par].dropna().values
        t_stat, p_val = stats.ttest_ind(exp, nov, equal_var=False)   # Welch
        u_stat, p_u = stats.mannwhitneyu(exp, nov, alternative='two-sided')
        d = pg.compute_effsize(exp, nov, eftype='cohen')
        res_indep.append({
            'Font': font, 'Parameter': par,
            'M_Novice': nov.mean(), 'M_Expert': exp.mean(),
            'Diff': exp.mean() - nov.mean(),
            't_welch': t_stat, 'p_welch': p_val,
            'U': u_stat, 'p_MWU': p_u, 'Cohens_d': d,
        })

df_indep_tests = pd.DataFrame(res_indep)
_, p_holm_i, _, _ = multipletests(df_indep_tests['p_welch'].values,
                                  alpha=0.05, method='holm')
_, p_bh_i, _, _ = multipletests(df_indep_tests['p_welch'].values,
                                alpha=0.05, method='fdr_bh')
df_indep_tests['p_holm'] = p_holm_i
df_indep_tests['p_BH'] = p_bh_i
print(df_indep_tests.to_string(index=False, float_format='{:.4f}'.format))

print("\n--- Side-by-side: hierarchical (shrunk) vs independent (unshrunk) ---")
print(f"  {'Font':>15s} {'Par':>3s} {'d hier':>8s} {'p hier':>8s} "
      f"{'d indep':>8s} {'p indep':>8s} {'p indep Holm':>13s}")
for font in fonts_list:
    for par in ['K', 'C', 't0']:
        h = df_ttest[(df_ttest['Font'] == font) & (df_ttest['Parameter'] == par)].iloc[0]
        i = df_indep_tests[(df_indep_tests['Font'] == font) &
                           (df_indep_tests['Parameter'] == par)].iloc[0]
        print(f"  {font:>15s} {par:>3s} {h['Cohens_d']:>8.3f} {h['p_value']:>8.4f} "
              f"{i['Cohens_d']:>8.3f} {i['p_welch']:>8.4f} {i['p_holm']:>13.4f}")

print("\n  The manuscript reports the INDEPENDENT column (Table 3, Section 9.7).")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S3d. Figure 4 — shrunk vs unshrunk parameter estimates")
print("=" * 78)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ri, font in enumerate(fonts_list):
    for ci, par in enumerate(['K', 'C', 't0']):
        ax = axes[ri, ci]
        s = cmp[cmp['font'] == font]
        for grp, color in [('Novice', '#1f77b4'), ('Expert', '#d62728')]:
            g = s[s['group'] == grp]
            ax.scatter(g[f'{par}_indep'], g[f'{par}_hier'], c=color, s=45,
                       alpha=0.75, edgecolors='black', linewidth=0.3, label=grp)
        lims = [min(s[f'{par}_indep'].min(), s[f'{par}_hier'].min()),
                max(s[f'{par}_indep'].max(), s[f'{par}_hier'].max())]
        ax.plot(lims, lims, 'k--', lw=1, alpha=0.6, label='identity')
        ax.set_xlabel(f'{par} — independent fit')
        ax.set_ylabel(f'{par} — hierarchical (shrunk)')
        ax.set_title(f'{font}: {par}', fontsize=11, fontweight='bold')
        ax.grid(alpha=0.25)
        if ri == 0 and ci == 0:
            ax.legend(fontsize=8)
fig.suptitle('Shrinkage check: points falling inside the identity line indicate '
             'pooling toward the group x font mean', fontsize=13, y=1.02)
fig.tight_layout()
savefig("Fig04_shrinkage_check")
plt.show()

### 9.4 Sensitivity refit with Expert #8 reinstated

In [ ]:
# 9.4 Sensitivity analysis: Expert #8 reinstated
# ============================================================================
# R8_Expert was removed to maintain the age balance between groups. Because
# that exclusion is data-dependent, and age turned out not to predict
# performance, the full hierarchical model is refitted with R8_Expert
# REINSTATED (keeping the exclusion of R17_Novice, a genuine ~100 ms timing
# failure) to check whether the conclusions move.
#
# Runtime note: this refits the full hierarchical model. It caches to its own
# NetCDF file, so it only samples once.

filename_sens = str(RESULTS_DIR / "posterior_full_with_R8.nc")

print("=" * 78)
print("S4a. Rebuild the analysis frame with R8_Expert reinstated")
print("=" * 78)


def build_dat_sensitivity(raw: pd.DataFrame) -> pd.DataFrame:
    """Same pipeline as build_analysis_data but excluding ONLY R17_Novice."""
    d = raw.copy()
    d = d.query("not (ParticipantID == 'R17' and GroupCondition == 'Novice')").reset_index(drop=True)

    nov_mask = d['GroupCondition'] == 'Novice'
    exp_mask = d['GroupCondition'] == 'Expert'

    d.loc[nov_mask, 'n_frames'] = np.round(
        d.loc[nov_mask, 'TargetDisplay_chrono'] * 60.0 / 1000.0)
    d.loc[nov_mask, 't_corrected_ms'] = d.loc[nov_mask, 'n_frames'] * FRAME_MS
    d.loc[exp_mask, 't_corrected_ms'] = d.loc[exp_mask, 'TargetDisplay_ms'].astype(float)

    novice_keep = pd.Series(False, index=d.index)
    for t_int, t_dom in DOMINANT_MAP.items():
        novice_keep |= (nov_mask &
                        (d['TargetDisplay_ms'] == t_int) &
                        (np.abs(d['t_corrected_ms'] - t_dom) < 0.1))
    d = d[novice_keep | exp_mask].reset_index(drop=True)

    d["t"] = d["t_corrected_ms"]
    d["t_intended"] = d["TargetDisplay_ms"]
    d["P"] = d["NumberCorrect"]
    d["S"] = 5
    d["participant"] = d["Participants_ID"]
    d["font"] = d["Font"]
    d["group"] = d["GroupCondition"]
    d["isKlimax"] = (d["font"] == "Klimax Plus").astype(int)
    d["isExpert"] = (d["group"] == "Expert").astype(int)
    d["t_s"] = d["t"] / 1000.0
    return d


dat_s = build_dat_sensitivity(dat_raw)
print(f"  Participants: {dat_s['participant'].nunique()} "
      f"(Novice {dat_s[dat_s['group']=='Novice']['participant'].nunique()}, "
      f"Expert {dat_s[dat_s['group']=='Expert']['participant'].nunique()})")
print(f"  Trials: {len(dat_s)}  (main analysis: {len(dat)})")

# Age balance with R8 back in (R17_Novice remains excluded)
ages_s = participant_info_all.loc[
    participant_info_all['Participants_ID'] != 'R17_Novice',
    ['Participants_ID', 'group', 'age']].copy()
ages_s['age'] = ages_s['age'].astype(int)

print("\n  Age by group WITH R8_Expert reinstated:")
print(ages_s.groupby('group')['age'].agg(['count', 'mean', 'std', 'min', 'max']).round(2).to_string())
_n = ages_s[ages_s['group'] == 'Novice']['age'].values
_e = ages_s[ages_s['group'] == 'Expert']['age'].values
_t, _p = stats.ttest_ind(_e, _n)
print(f"  Age difference: t = {_t:.3f}, p = {_p:.4f}, "
      f"d = {pg.compute_effsize(_e, _n, eftype='cohen'):+.3f}")
print("  -> If the groups remain age-matched with R8 included, the original")
print("     exclusion was unnecessary.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S4b. Refit the full hierarchical model on the sensitivity sample")
print("=" * 78)

parts_s = dat_s['participant'].astype('category')
pidx_s = parts_s.cat.codes.values
pnames_s = list(parts_s.cat.categories)
n_parts_s = len(pnames_s)
pgroup_s = (dat_s.groupby('participant')['isExpert'].first()
            .reindex(pnames_s).values.astype(int))

y_s = dat_s['P'].to_numpy()
ts_s = dat_s['t_s'].to_numpy()
S_s = dat_s['S'].to_numpy()
font_idx_s = dat_s['isKlimax'].values

with pm.Model() as m_sens:
    gm_klin = pm.Normal('group_mean_klin', mu=mu_klin, sigma=1.5, shape=n_groups)
    gm_clin = pm.Normal('group_mean_clin', mu=mu_clin, sigma=1.0, shape=n_groups)
    gm_tlin = pm.Normal('group_mean_tlin', mu=mu_tlin, sigma=0.7, shape=n_groups)

    sd_k = pm.HalfNormal('sd_klin_part', sigma=0.7)
    sd_c = pm.HalfNormal('sd_clin_part', sigma=0.7)
    sd_t = pm.HalfNormal('sd_tlin_part', sigma=0.7)

    zk = pm.Normal('z_klin_part', 0, 1, shape=n_parts_s)
    zc = pm.Normal('z_clin_part', 0, 1, shape=n_parts_s)
    zt = pm.Normal('z_tlin_part', 0, 1, shape=n_parts_s)

    part_k = pm.Deterministic('part_klin', gm_klin[pgroup_s] + sd_k * zk)
    part_c = pm.Deterministic('part_clin', gm_clin[pgroup_s] + sd_c * zc)
    part_t = pm.Deterministic('part_tlin', gm_tlin[pgroup_s] + sd_t * zt)

    f_klin = pm.Normal('font_klin', mu=0, sigma=0.7)
    f_clin = pm.Normal('font_clin', mu=0, sigma=0.7)
    f_tlin = pm.Normal('font_tlin', mu=0, sigma=0.5)

    sdf_k = pm.HalfNormal('sd_font_klin', sigma=0.3)
    sdf_c = pm.HalfNormal('sd_font_clin', sigma=0.3)
    sdf_t = pm.HalfNormal('sd_font_tlin', sigma=0.2)

    zf_k = pm.Normal('z_font_klin', 0, 1, shape=n_parts_s)
    zf_c = pm.Normal('z_font_clin', 0, 1, shape=n_parts_s)
    zf_t = pm.Normal('z_font_tlin', 0, 1, shape=n_parts_s)

    fg_k = pm.Normal('fontgroup_klin', mu=0, sigma=0.5)
    fg_c = pm.Normal('fontgroup_clin', mu=0, sigma=0.5)
    fg_t = pm.Normal('fontgroup_tlin', mu=0, sigma=0.3)

    isK_obs = font_idx_s
    isE_obs = pgroup_s[pidx_s]

    klin = part_k[pidx_s] + (f_klin + sdf_k * zf_k[pidx_s]) * isK_obs + fg_k * isK_obs * isE_obs
    clin = part_c[pidx_s] + (f_clin + sdf_c * zf_c[pidx_s]) * isK_obs + fg_c * isK_obs * isE_obs
    tlin = part_t[pidx_s] + (f_tlin + sdf_t * zf_t[pidx_s]) * isK_obs + fg_t * isK_obs * isE_obs

    K = S_s * pm.math.sigmoid(klin)
    C = pm.math.exp(clin)
    t0 = pm.math.exp(tlin)

    K_safe = pm.math.clip(K, 1e-12, np.inf)
    pos_dt = pos_stable(ts_s - t0, eps=1e-12)
    expo = pm.math.clip(-(C / K_safe) * pos_dt, -1e6, 1e6)
    mu_s = K * (1 - pm.math.exp(expo))

    sigma = pm.HalfStudentT('sigma', nu=3, sigma=2)
    nu = pm.Gamma('nu', alpha=2, beta=0.1)
    pm.StudentT('P_obs', nu=nu, mu=mu_s, sigma=sigma, observed=y_s)

if os.path.exists(filename_sens):
    print(f"  Loading cached sensitivity fit from {filename_sens}...")
    idata_sens = az.from_netcdf(filename_sens)
else:
    print("  Sampling sensitivity model (this takes a while)...")
    with m_sens:
        idata_sens = pm.sample(draws=3000, tune=1500, **SAMPLE_KWARGS)
    idata_sens.to_netcdf(filename_sens)
    print(f"  Saved to {filename_sens}.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S4c. Do the conclusions change?")
print("=" * 78)

post_s = idata_sens.posterior
_s_gm_klin = _extract_vec(post_s, "group_mean_klin")
_s_gm_clin = _extract_vec(post_s, "group_mean_clin")
_s_gm_tlin = _extract_vec(post_s, "group_mean_tlin")
_s_f_klin = _extract_1d(post_s, "font_klin")
_s_f_clin = _extract_1d(post_s, "font_clin")
_s_f_tlin = _extract_1d(post_s, "font_tlin")
_s_fg_klin = _extract_1d(post_s, "fontgroup_klin")
_s_fg_clin = _extract_1d(post_s, "fontgroup_clin")
_s_fg_tlin = _extract_1d(post_s, "fontgroup_tlin")


def get_KCt0_sens(font, group):
    g = 0 if group == "Novice" else 1
    isK = 1.0 if font == "Klimax Plus" else 0.0
    isE = 1.0 if group == "Expert" else 0.0
    klin = _s_gm_klin[:, g] + _s_f_klin * isK + _s_fg_klin * isK * isE
    clin = _s_gm_clin[:, g] + _s_f_clin * isK + _s_fg_clin * isK * isE
    tlin = _s_gm_tlin[:, g] + _s_f_tlin * isK + _s_fg_tlin * isK * isE
    return 5.0 * sigmoid(klin), np.exp(clin), np.exp(tlin)


print(f"\n  {'Contrast':<44s} {'main: mean [95% CrI], P(>0)':<34s} "
      f"{'+R8: mean [95% CrI], P(>0)'}")
for par_i, par in enumerate(['K', 'C', 't0']):
    for font in fonts_list:
        de_main = get_KCt0(font, 'Expert')[par_i] - get_KCt0(font, 'Novice')[par_i]
        de_sens = get_KCt0_sens(font, 'Expert')[par_i] - get_KCt0_sens(font, 'Novice')[par_i]
        if par == 't0':
            de_main, de_sens = de_main * 1000, de_sens * 1000
        lm, hm = np.quantile(de_main, [0.025, 0.975])
        ls, hs = np.quantile(de_sens, [0.025, 0.975])
        label = f"{par}: Expert-Novice ({font})"
        print(f"  {label:<44s} {de_main.mean():>7.3f} [{lm:>7.3f},{hm:>7.3f}] "
              f"{(de_main>0).mean():.3f}   "
              f"{de_sens.mean():>7.3f} [{ls:>7.3f},{hs:>7.3f}] {(de_sens>0).mean():.3f}")

print("\n  Font x Group interaction (K), main vs +R8:")
i_main = ((get_KCt0('Klimax Plus', 'Expert')[0] - get_KCt0('Klimax Plus', 'Novice')[0]) -
          (get_KCt0('Helvetica', 'Expert')[0] - get_KCt0('Helvetica', 'Novice')[0]))
i_sens = ((get_KCt0_sens('Klimax Plus', 'Expert')[0] - get_KCt0_sens('Klimax Plus', 'Novice')[0]) -
          (get_KCt0_sens('Helvetica', 'Expert')[0] - get_KCt0_sens('Helvetica', 'Novice')[0]))
print(f"    main: {i_main.mean():+.3f} items, P(>0) = {(i_main>0).mean():.3f}")
print(f"    +R8 : {i_sens.mean():+.3f} items, P(>0) = {(i_sens>0).mean():.3f}")

print("\n  -> If the two columns agree, the conclusions are unchanged with")
print("     R8_Expert included.")

### 9.5 DTW uncertainty, noise floor and temporal lead (Figure 10)

In [ ]:
# 9.5 DTW uncertainty, noise floor and the temporal lead (Figure 10)
# ============================================================================
# Three points about the DTW distance computed in Section 8:
#
#  (1) A bare distance carries no uncertainty: S5a gives a participant
#      bootstrap CI and S5b a within-group split-half noise floor.
#  (2) The larger distance for Helvetica than Klimax Plus is a scale effect:
#      `normalizedDistance` is the mean absolute VERTICAL residual along the
#      warping path, in units of letters correct. Helvetica performance is
#      roughly 1.6x higher overall, so the same relative separation produces a
#      larger absolute residual. Dividing by the mean performance level removes
#      almost all of the apparent font difference (S5c).
#  (3) DTW distance is an AMPLITUDE statistic. It does not measure a temporal
#      lead. The claim of interest ("experts reach the same performance at
#      shorter exposures") is a HORIZONTAL shift, so S5d estimates that
#      directly, with bootstrap intervals.
#
# A label-permutation test is NOT available here: the exposure-timing regime
# is perfectly confounded with group (the novices ran on the miscalibrated
# setup), so shuffling group labels would also shuffle the time axis and
# destroy the quantity being tested. Participant bootstrap intervals plus a
# within-group split-half noise floor are reported instead, and the DTW itself
# is treated as descriptive.

rng_boot = np.random.default_rng(20260806)
N_BOOT = 2000

# ---------------------------------------------------------------------------
# Precompute, for each font x group, a (participant x time) matrix of trial
# SUMS and COUNTS. Bootstrapping participants is then pure numpy: no repeated
# DataFrame slicing, and the estimator stays identical to the observed curve
# (trial-weighted mean P at each exposure time).
# ---------------------------------------------------------------------------
_panel = {}
for font in fonts_list:
    for grp in groups_list:
        sub = dat[(dat['font'] == font) & (dat['group'] == grp)]
        times = np.array(sorted(sub['t'].unique()))
        pids = np.array(sorted(sub['participant'].unique()))
        t_pos = {t: j for j, t in enumerate(times)}
        p_pos = {p: i for i, p in enumerate(pids)}
        Ssum = np.zeros((len(pids), len(times)))
        Cnt = np.zeros((len(pids), len(times)))
        for (p, t), g in sub.groupby(['participant', 't']):
            Ssum[p_pos[p], t_pos[t]] = g['P'].sum()
            Cnt[p_pos[p], t_pos[t]] = len(g)
        _panel[(font, grp)] = dict(times=times, pids=pids, s=Ssum, n=Cnt)


def _curve_np(pan, rows=None):
    """Trial-weighted mean P per exposure time for a subset of participant rows."""
    s = pan['s'] if rows is None else pan['s'][rows]
    n = pan['n'] if rows is None else pan['n'][rows]
    tot_n = n.sum(axis=0)
    keep = tot_n > 0
    return pan['times'][keep], (s.sum(axis=0)[keep] / tot_n[keep])


def _dtw_dist(a, b):
    return dtw_align(np.asarray(a, float), np.asarray(b, float)).normalizedDistance


print("=" * 78)
print("S5a. Bootstrap CI for the DTW distance (resampling participants)")
print("=" * 78)

dtw_boot = {}
for font in fonts_list:
    pn, pe = _panel[(font, 'Novice')], _panel[(font, 'Expert')]
    _, cn_obs = _curve_np(pn)
    _, ce_obs = _curve_np(pe)
    obs = _dtw_dist(cn_obs, ce_obs)

    boots = np.empty(N_BOOT)
    boots[:] = np.nan
    for b in range(N_BOOT):
        rn = rng_boot.integers(0, len(pn['pids']), len(pn['pids']))
        re_ = rng_boot.integers(0, len(pe['pids']), len(pe['pids']))
        _, cb_n = _curve_np(pn, rn)
        _, cb_e = _curve_np(pe, re_)
        if len(cb_n) >= 3 and len(cb_e) >= 3:
            boots[b] = _dtw_dist(cb_n, cb_e)
    boots = boots[np.isfinite(boots)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    dtw_boot[font] = dict(obs=obs, boots=boots, lo=lo, hi=hi)
    print(f"  {font:>15s}: DTW = {obs:.4f}, "
          f"bootstrap 95% CI [{lo:.4f}, {hi:.4f}]  (B = {len(boots)})")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S5b. Within-group split-half noise floor")
print("=" * 78)
print("  How large a DTW distance arises between two random halves of the SAME")
print("  group (same population, same timing regime)? If the between-group")
print("  distance sits inside that distribution, the DTW is not distinguishing")
print("  the groups at all.\n")

for font in fonts_list:
    floors = []
    for grp in groups_list:
        pan = _panel[(font, grp)]
        m = len(pan['pids'])
        for _ in range(N_BOOT // 2):
            perm = rng_boot.permutation(m)
            h1, h2 = perm[:m // 2], perm[m // 2:]
            _, c1 = _curve_np(pan, h1)
            _, c2 = _curve_np(pan, h2)
            if len(c1) >= 3 and len(c2) >= 3:
                floors.append(_dtw_dist(c1, c2))
    floors = np.asarray(floors)
    obs = dtw_boot[font]['obs']
    pct = float((floors >= obs).mean())
    print(f"  {font:>15s}: split-half floor median = {np.median(floors):.4f}, "
          f"95th pct = {np.percentile(floors, 95):.4f}")
    print(f"  {'':>15s}  observed between-group DTW = {obs:.4f}; "
          f"proportion of split-half distances >= observed = {pct:.3f}")
    verdict = ("NOT distinguishable from within-group sampling noise"
               if pct > 0.05 else "exceeds the within-group noise floor")
    print(f"  {'':>15s}  -> {verdict}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S5c. Why Helvetica > Klimax Plus: DTW distance is scale-dependent")
print("=" * 78)
print(f"  {'Font':>15s} {'raw DTW':>9s} {'mean P':>8s} {'DTW / mean P':>13s}")
for font in fonts_list:
    _, cn = _curve_np(_panel[(font, 'Novice')])
    _, ce = _curve_np(_panel[(font, 'Expert')])
    mean_level = np.concatenate([cn, ce]).mean()
    raw = dtw_boot[font]['obs']
    print(f"  {font:>15s} {raw:>9.4f} {mean_level:>8.3f} {raw/mean_level:>13.4f}")
print("\n  Expressed relative to the performance level the two fonts give")
print("  near-identical values. The raw ordering reflects the height of the")
print("  Helvetica curves, not a stronger expert lead in the easy font.")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S5d. The statistic that actually tests the claim: horizontal lead")
print("=" * 78)
print("  For criterion c letters, t_c(group) = the exposure at which the")
print("  group's empirical curve first reaches c (linear interpolation). The")
print("  expert lead is t_c(Novice) - t_c(Expert); the ratio is t_c(N)/t_c(E).\n")


def time_to_criterion(times, p, c):
    """Linear interpolation of the first crossing of level c. NaN if never reached."""
    p = np.asarray(p, float)
    times = np.asarray(times, float)
    if p.size == 0 or p.max() < c:
        return np.nan
    for i in range(1, len(p)):
        if p[i] >= c:
            if p[i] == p[i - 1]:
                return times[i]
            return times[i - 1] + (c - p[i - 1]) / (p[i] - p[i - 1]) * (times[i] - times[i - 1])
    return np.nan


CRITERIA = [1.0, 2.0, 3.0]
lead_rows = []
for font in fonts_list:
    pn, pe = _panel[(font, 'Novice')], _panel[(font, 'Expert')]
    tn_g, cn_obs = _curve_np(pn)
    te_g, ce_obs = _curve_np(pe)

    # one bootstrap loop, all criteria evaluated per replicate
    boot_tn = {c: [] for c in CRITERIA}
    boot_te = {c: [] for c in CRITERIA}
    for _ in range(N_BOOT):
        rn = rng_boot.integers(0, len(pn['pids']), len(pn['pids']))
        re_ = rng_boot.integers(0, len(pe['pids']), len(pe['pids']))
        tnb, cnb = _curve_np(pn, rn)
        teb, ceb = _curve_np(pe, re_)
        for c in CRITERIA:
            boot_tn[c].append(time_to_criterion(tnb, cnb, c))
            boot_te[c].append(time_to_criterion(teb, ceb, c))

    for c in CRITERIA:
        a = np.asarray(boot_tn[c], float)
        b = np.asarray(boot_te[c], float)
        ok = np.isfinite(a) & np.isfinite(b) & (b > 0)
        diffs, ratios = a[ok] - b[ok], a[ok] / b[ok]
        tn = time_to_criterion(tn_g, cn_obs, c)
        te = time_to_criterion(te_g, ce_obs, c)
        lead_rows.append(dict(
            Font=font, criterion=c, t_novice=tn, t_expert=te,
            lead_ms=(tn - te) if np.isfinite(tn) and np.isfinite(te) else np.nan,
            lead_lo=np.percentile(diffs, 2.5) if diffs.size else np.nan,
            lead_hi=np.percentile(diffs, 97.5) if diffs.size else np.nan,
            ratio=(tn / te) if np.isfinite(tn) and np.isfinite(te) else np.nan,
            ratio_lo=np.percentile(ratios, 2.5) if ratios.size else np.nan,
            ratio_hi=np.percentile(ratios, 97.5) if ratios.size else np.nan,
            P_lead_gt0=float((diffs > 0).mean()) if diffs.size else np.nan,
            pct_boot_defined=100.0 * ok.mean()))

df_lead = pd.DataFrame(lead_rows)
df_lead.to_csv(TABLE_DIR / "dtw_temporal_lead.csv", index=False)
print(df_lead.to_string(index=False, float_format='{:.3f}'.format))
print("\n  Rows whose bootstrap CI on the lead includes 0 provide no evidence of")
print("  a temporal lead at that criterion. A low pct_boot_defined means the")
print("  criterion was often out of range (e.g. Klimax Plus never reaches 3")
print("  within the novice exposure range), so that row is uninterpretable.")
print("\n  CAVEAT: t_novice is read off the FRAME-SNAPPED novice axis. If the")
print("  snap over-corrects, the lead is overstated; if it under-corrects, it")
print("  is understated. Read alongside Section 9.1 (S1c).")

# ---------------------------------------------------------------------------
# Figure: green = Helvetica, orange = Klimax Plus in every panel (same colours as the other figures)
font_colors  = {"Helvetica": "#1b7837", "Klimax Plus": "#f28e2b"}
font_markers = {"Helvetica": "s", "Klimax Plus": "o"}

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

ax = axes[0]
for font in fonts_list:
    b = dtw_boot[font]
    ax.hist(b['boots'], bins=45, alpha=0.55, density=True, color=font_colors[font],
            label=f"{font} (obs {b['obs']:.3f})")
    ax.axvline(b['obs'], color=font_colors[font], lw=2)
ax.set_xlabel('Normalised DTW distance')
ax.set_ylabel('Density')
ax.set_title('Bootstrap distribution of the DTW distance', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

ax = axes[1]
# Omit criteria with no observed lead (Klimax Plus at 3 letters: the novice curve never reaches it)
df_plot = df_lead[np.isfinite(df_lead['lead_ms'])].reset_index(drop=True)
xs = np.arange(len(df_plot))
for font in fonts_list:
    m = (df_plot['Font'] == font).to_numpy()
    d = df_plot[m]
    ax.errorbar(xs[m], d['lead_ms'],
                yerr=[d['lead_ms'] - d['lead_lo'], d['lead_hi'] - d['lead_ms']],
                fmt=font_markers[font], capsize=4, color=font_colors[font])
ax.axhline(0, color='grey', ls='--', lw=1)
ax.set_xticks(xs)
ax.set_xticklabels([f"{r['Font']}\n{r['criterion']:.0f} letter" + ("s" if r['criterion'] > 1 else "")
                    for _, r in df_plot.iterrows()], fontsize=8)
ax.set_xlim(-0.5, len(df_plot) - 0.5)
ax.set_ylabel('Expert temporal lead (ms)')
ax.set_title('Time-to-criterion lead, bootstrap 95% CI', fontsize=11, fontweight='bold')
ax.grid(alpha=0.25, axis='y')

ax = axes[2]
for font in fonts_list:
    color, marker = font_colors[font], font_markers[font]
    tn_g, cn_obs = _curve_np(_panel[(font, 'Novice')])
    te_g, ce_obs = _curve_np(_panel[(font, 'Expert')])
    ax.plot(tn_g, cn_obs, marker + '-', color=color, alpha=0.55,
            label=f'{font} Novice')
    ax.plot(te_g, ce_obs, marker + '--', color=color,
            label=f'{font} Expert')
for c in CRITERIA:
    ax.axhline(c, color='grey', ls=':', lw=0.8)
ax.set_xlabel('Exposure (ms; novice = frame-snapped)')
ax.set_ylabel('Mean letters correct')
ax.set_title('Empirical curves with criterion levels', fontsize=11, fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.25)

fig.tight_layout()
savefig("Fig10_dtw_bootstrap_temporal_lead")
plt.show()

### 9.6 Sensitivity power analysis, trials per cell, group composition and stimulus statistics

In [ ]:
# 9.6 Sensitivity power analysis, trials per cell, group composition, stimulus statistics
# ============================================================================

from statsmodels.stats.power import TTestIndPower

print("=" * 78)
print("S6a. SENSITIVITY POWER ANALYSIS (post hoc)")
print("=" * 78)
print("  The study was not pre-registered and no a priori power analysis was")
print("  run. The honest substitute is a SENSITIVITY analysis: given the")
print("  realised n, what is the smallest effect the design could detect?\n")

pw = TTestIndPower()
n_per_group = 17
for power in [0.80, 0.90]:
    d_min = pw.solve_power(effect_size=None, nobs1=n_per_group, alpha=0.05,
                           power=power, ratio=1.0, alternative='two-sided')
    print(f"  n = {n_per_group}/group, alpha = .05 two-tailed, power = {power:.0%}"
          f"  ->  smallest detectable d = {d_min:.2f}")

print("\n  Achieved power for the effects actually observed:")
for _, r in df_ttest.iterrows():
    ach = pw.solve_power(effect_size=abs(r['Cohens_d']), nobs1=n_per_group,
                         alpha=0.05, ratio=1.0, alternative='two-sided')
    fs = 'Helvetica' if 'Helv' in r['Font'] else 'Klimax Plus'
    print(f"    {fs:>9s} {r['Parameter']:>2s}: d = {r['Cohens_d']:+.2f} "
          f"-> power = {ach:.2f}")

print("\n  -> With 17 per group the design is only well powered for LARGE")
print("     effects (d ~ 1.0). Null results, in particular the Helvetica")
print("     contrasts, cannot be read as evidence of equivalence.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S6b. TRIALS PER CELL")
print("=" * 78)
cellcount = (dat.groupby(['participant', 'font', 't'])
             .size().rename('n').reset_index())
print(f"  Trials per participant x font x duration: "
      f"median = {cellcount['n'].median():.0f}, "
      f"min = {cellcount['n'].min()}, max = {cellcount['n'].max()}")
print("\n  By group:")
for grp in ['Novice', 'Expert']:
    ids = dat[dat['group'] == grp]['participant'].unique()
    s = cellcount[cellcount['participant'].isin(ids)]['n']
    print(f"    {grp:>6s}: median {s.median():.0f} "
          f"(range {s.min()}-{s.max()}) across {len(s)} cells")
print("\n  Design intent: 160 trials = 2 fonts x 8 durations x 10 trials.")
print("  10 observations per point is thin for individually stable C in")
print("  particular.")

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S6c. GROUP COMPOSITION")
print("=" * 78)
print("  Occupation is not recorded in data/participant_info.csv; the composition")
print("  of the novice group is documented in the Methods section of the manuscript.")

print("\n  Vision status (both groups):")
print(participant_info.groupby(['group', 'vision']).size().to_string())
print("\n  Gender:")
print(participant_info.groupby(['group', 'gender']).size().to_string())

# --------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S6d. STIMULUS / DISPLAY PARAMETERS RECOVERABLE FROM THE DATA")
print("=" * 78)
print("  (screen size, pixel density, viewing distance and rendering settings are")
print("   not in the data files; see the Methods section of the manuscript.)\n")

print("  Rendered size per font (FontSize field, matched on x-height):")
print(dat_raw.groupby('Font')['FontSize'].agg(['min', 'max', 'nunique']).to_string())

print("\n  Word condition (pseudoword vs random string):")
print(dat_raw['WordCondition'].value_counts().to_string())

letters_all = dat_raw['Letters'].astype(str)
alphabet = sorted(set(''.join(letters_all)))
print(f"\n  String length: {sorted(letters_all.str.len().unique())}")
print(f"  Alphabet used ({len(alphabet)} distinct characters): {''.join(alphabet)}")
rep = letters_all.apply(lambda s: len(set(s)) < len(s))
print(f"  Strings containing a repeated letter: {int(rep.sum())} / {len(letters_all)}")
print(f"  Distinct strings presented: {letters_all.nunique()}")

from collections import Counter
cnt = Counter(''.join(letters_all))
freqs = np.array([cnt[a] for a in alphabet])
print(f"  Per-letter frequency across all trials: mean = {freqs.mean():.1f}, "
      f"sd = {freqs.std():.1f}, min = {freqs.min()}, max = {freqs.max()}")

if 'MaskDisplay_chrono' in dat_raw.columns:
    print("\n  Mask timing field (MaskDisplay_chrono) summary:")
    print(dat_raw.groupby('GroupCondition')['MaskDisplay_chrono']
          .agg(['mean', 'std', 'min', 'max']).round(2).to_string())
    print("  (recorded mask duration; nominal 800 ms)")

### 9.7 Reconciling the parameter estimands; Table 3 (independent fits); Figure S9

In [ ]:
# 9.7 Reconciling the parameter estimands; Table 3 (independent fits); Figure S9
# ============================================================================
# Four different quantities can all be called "mean K for Novice-Helvetica",
# and they legitimately differ:
#
#   E1  POPULATION-LEVEL POSTERIOR MEAN  (the forest plot, Figure 6)
#         E_s[ 5 * sigmoid( group_mean_klin[g]^(s) + font^(s) + fg^(s) ) ]
#       This is the capacity of a hypothetical participant sitting exactly at
#       the group mean of the LATENT scale, i.e. with a zero random effect.
#
#   E2  POSTERIOR MEAN OF THE GROUP-AVERAGE PARTICIPANT CAPACITY  (computed here)
#         E_s[ (1/n) * sum_i 5 * sigmoid( kappa_i^(s) ) ]
#       This is what a reader means by "the average capacity of this group".
#
#   E3  AVERAGE OF PLUG-IN PARTICIPANT ESTIMATES  (Section 5.2)
#         (1/n) * sum_i 5 * sigmoid( E_s[kappa_i^(s)] )      <- transform of mean
#
#   E4  AVERAGE OF INDEPENDENT, UNPOOLED PER-PARTICIPANT FITS  (Section 9.3)
#
# E1 vs E2/E3 is NOT a bug. K = 5*sigmoid(kappa) is concave for kappa > 0, and
# the between-participant SD on the latent scale is large (sd_klin_part ~ 0.78).
# By Jensen's inequality the average of individual capacities is therefore BELOW
# the capacity of the average participant. The same logic runs the other way for
# C = exp(clin), which is convex, so there the group average sits ABOVE the
# population-level value.
#
# E2 vs E3 IS an estimator artefact: E3 transforms posterior means instead of
# taking the posterior mean of the transform, and it multiplies posterior means
# (sd_font * z) rather than taking the mean of the product. E2 is correct.
#
# The manuscript reports E1 in the forest plot (labelled as a population-level
# posterior) and E4 in the parameter table (Table 3, labelled as independent
# per-participant fits); E3 is not reported.

print("=" * 78)
print("S7a. Draw-wise participant-level parameters (the correct E2 estimator)")
print("=" * 78)

_p = idata_full.posterior


def _vec(name):
    da = _p[name].stack(sample=("chain", "draw"))
    if da.dims[0] != "sample":
        da = da.transpose("sample", ...)
    return da.values


# (n_draws, n_participants)
pk = _vec("part_klin")
pc = _vec("part_clin")
pt = _vec("part_tlin")
zfk, zfc, zft = _vec("z_font_klin"), _vec("z_font_clin"), _vec("z_font_tlin")
# (n_draws,)
fk, fc, ft = _vec("font_klin"), _vec("font_clin"), _vec("font_tlin")
sfk, sfc, sft = _vec("sd_font_klin"), _vec("sd_font_clin"), _vec("sd_font_tlin")
gk, gc, gt = _vec("fontgroup_klin"), _vec("fontgroup_clin"), _vec("fontgroup_tlin")

isE_vec = participant_group.astype(float)          # (n_participants,)
n_draws = pk.shape[0]

drawwise = {}
for font_name, isK in [("Helvetica", 0.0), ("Klimax Plus", 1.0)]:
    kap = pk + (fk[:, None] + sfk[:, None] * zfk) * isK + gk[:, None] * isK * isE_vec[None, :]
    cln = pc + (fc[:, None] + sfc[:, None] * zfc) * isK + gc[:, None] * isK * isE_vec[None, :]
    tln = pt + (ft[:, None] + sft[:, None] * zft) * isK + gt[:, None] * isK * isE_vec[None, :]
    drawwise[font_name] = {
        "K": 5.0 / (1.0 + np.exp(-kap)),          # (n_draws, n_participants)
        "C": np.exp(cln),
        "t0": np.exp(tln) * 1000.0,               # ms
    }

# E2 per participant = posterior mean of the transformed quantity
rows_dw = []
for font_name in fonts_list:
    for pi, pname in enumerate(participant_names):
        rows_dw.append({
            "participant": pname,
            "group": "Expert" if participant_group[pi] == 1 else "Novice",
            "font": font_name,
            "K": drawwise[font_name]["K"][:, pi].mean(),
            "C": drawwise[font_name]["C"][:, pi].mean(),
            "t0": drawwise[font_name]["t0"][:, pi].mean() / 1000.0,   # back to s
        })
df_kct_drawwise = pd.DataFrame(rows_dw)
print(f"  Computed for {len(df_kct_drawwise)} participant x font cells "
      f"from {n_draws} draws.")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S7b. THE RECONCILIATION TABLE (population-level vs participant-level estimands)")
print("=" * 78)

recon_rows = []
for par, unit, scale in [("K", "items", 1.0), ("C", "items/s", 1.0), ("t0", "ms", 1000.0)]:
    for font in fonts_list:
        for grp in groups_list:
            gi = 0 if grp == "Novice" else 1
            gidx = np.array([i for i, pn in enumerate(participant_names)
                             if participant_group[i] == gi])

            # E1 population-level
            K1, C1, T1 = get_KCt0(font, grp)
            e1 = {"K": K1, "C": C1, "t0": T1 * 1000.0}[par].mean()

            # E2 posterior mean of the group-average participant value
            M = drawwise[font][par][:, gidx]        # (draws, n_in_group)
            e2_draws = M.mean(axis=1)
            e2 = e2_draws.mean()
            e2_lo, e2_hi = np.quantile(e2_draws, [0.025, 0.975])

            # E3 plug-in average (transform of posterior means)
            s3 = df_kct[(df_kct["font"] == font) & (df_kct["group"] == grp)][par]
            e3 = s3.mean() * scale

            # E4 independent unpooled fits
            s4 = df_indep[(df_indep["font"] == font) & (df_indep["group"] == grp)][par]
            e4 = s4.mean() * scale

            recon_rows.append({
                "Parameter": par, "Unit": unit, "Font": font, "Group": grp,
                "E1_population": e1, "E2_drawwise_mean": e2,
                "E2_lo": e2_lo, "E2_hi": e2_hi,
                "E3_plugin": e3, "E4_independent": e4,
                "E1_minus_E2": e1 - e2, "E2_minus_E3": e2 - e3,
            })

df_recon = pd.DataFrame(recon_rows)
pd.set_option("display.width", 200)
print(df_recon.to_string(index=False, float_format="{:.3f}".format))
df_recon.to_csv(TABLE_DIR / "estimand_reconciliation.csv", index=False)

print("\n  Read the two difference columns:")
print("   * E1_minus_E2 is JENSEN. It is positive for K (concave link: the mean")
print("     of individual capacities sits below the capacity of the mean")
print("     participant) and negative for C (convex link). It is a property of")
print("     the non-linear link, not an error.")
print("   * E2_minus_E3 is the PLUG-IN ARTEFACT and should be small.")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S7c. Table 3: parameter table from the INDEPENDENT fits (E4), Welch t-tests")
print("=" * 78)
print("  No partial pooling, hence no circularity.\n")

from scipy import stats as _st
from statsmodels.stats.multitest import multipletests

rows_tab = []
for font in fonts_list:
    for par, scale, unit in [("K", 1.0, "items"), ("C", 1.0, "items/s"),
                             ("t0", 1000.0, "ms")]:
        nov = df_indep[(df_indep["font"] == font) &
                       (df_indep["group"] == "Novice")][par].dropna().values * scale
        exp = df_indep[(df_indep["font"] == font) &
                       (df_indep["group"] == "Expert")][par].dropna().values * scale
        t_w, p_w = _st.ttest_ind(exp, nov, equal_var=False)
        n1, n2 = len(exp), len(nov)
        var_exp, var_nov = exp.var(ddof=1), nov.var(ddof=1)
        se = np.sqrt(var_exp / n1 + var_nov / n2)
        df_w = ((var_exp / n1 + var_nov / n2) ** 2
                / ((var_exp / n1) ** 2 / (n1 - 1) + (var_nov / n2) ** 2 / (n2 - 1)))
        tc = _st.t.ppf(0.975, df_w)
        diff = exp.mean() - nov.mean()
        rows_tab.append(dict(Font=font, Par=par, Unit=unit,
                             M_nov=nov.mean(), SD_nov=nov.std(ddof=1),
                             M_exp=exp.mean(), SD_exp=exp.std(ddof=1),
                             diff=diff, lo=diff - tc * se, hi=diff + tc * se,
                             t=t_w, df=df_w, p=p_w,
                             d=pg.compute_effsize(exp, nov, eftype="cohen"),
                             var_ratio=var_nov / var_exp))

df_tab = pd.DataFrame(rows_tab)
_, df_tab["p_holm"], _, _ = multipletests(df_tab["p"].values, alpha=0.05, method="holm")
print(df_tab.to_string(index=False, float_format="{:.4f}".format))
df_tab.to_csv(TABLE_DIR / "Table3_independent_fits.csv", index=False)

print("\n--- LaTeX body rows ---")
name = {"Helvetica": "Helvetica", "Klimax Plus": "Klimax Plus"}
lab = {"K": "$K$ (items)", "C": "$C$ (items/s)", "t0": "$t_0$ (ms)"}
for _, r in df_tab.iterrows():
    fmt = "{:.2f}" if r["Par"] != "C" else "{:.1f}"
    print(f"{name[r['Font']]} & {lab[r['Par']]} & "
          f"{fmt.format(r['M_nov'])} ({fmt.format(r['SD_nov'])}) & "
          f"{fmt.format(r['M_exp'])} ({fmt.format(r['SD_exp'])}) & "
          f"${r['diff']:+.2f}$ & [{r['lo']:+.2f}, {r['hi']:+.2f}] & "
          f"{r['t']:+.2f} & {r['df']:.1f} & {r['p']:.3f} & {r['p_holm']:.3f} & "
          f"${r['d']:+.2f}$ \\\\")

print("\n  Variance ratio (Novice/Expert) per cell — note how much more")
print("  heterogeneous the novice group is; this is why Welch is used:")
for _, r in df_tab.iterrows():
    print(f"    {r['Font']:>15s} {r['Par']:>2s}: {r['var_ratio']:.2f}x")

# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print("S7d. Figure S9: parameter box plots rebuilt from the INDEPENDENT fits")
print("=" * 78)

group_colors = {"Novice": "#1f77b4", "Expert": "#d62728"}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ylabels = ["$K$ (VSTM capacity, items)", "$C$ (processing speed, items/s)",
           "$t_0$ (perceptual threshold, ms)"]

for ax, par, ylab in zip(axes, ["K", "C", "t0"], ylabels):
    scale = 1000.0 if par == "t0" else 1.0
    for fi, font in enumerate(fonts_list):
        for gi, grp in enumerate(groups_list):
            x = fi * 2.5 + gi * 1.0
            sub = df_indep[(df_indep["font"] == font) &
                           (df_indep["group"] == grp)][par] * scale
            bp = ax.boxplot([sub], positions=[x], widths=0.7, patch_artist=True,
                            showmeans=True,
                            meanprops=dict(marker="D", markerfacecolor="white",
                                           markersize=6))
            bp["boxes"][0].set_facecolor(group_colors[grp])
            bp["boxes"][0].set_alpha(0.5)
            ax.scatter(np.random.default_rng(fi * 10 + gi).normal(x, 0.09, len(sub)),
                       sub, color=group_colors[grp], alpha=0.65, s=22, zorder=3)

    for fi, font in enumerate(fonts_list):
        r = df_tab[(df_tab["Font"] == font) & (df_tab["Par"] == par)].iloc[0]
        x1, x2 = fi * 2.5, fi * 2.5 + 1.0
        vals = df_indep[df_indep["font"] == font][par] * scale
        y_bar = vals.max() * 1.08
        ax.plot([x1, x1, x2, x2],
                [y_bar * 0.98, y_bar, y_bar, y_bar * 0.98], "k-", lw=1)
        ax.text((x1 + x2) / 2, y_bar * 1.01,
                f"p={r['p']:.3f}\n$p_{{Holm}}$={r['p_holm']:.3f}",
                ha="center", fontsize=8)

    ax.set_xticks([fi * 2.5 + 0.5 for fi in range(len(fonts_list))])
    ax.set_xticklabels(list(fonts_list))
    ax.set_ylabel(ylab)
    ax.set_title(par, fontsize=14, fontweight="bold")
    ax.grid(alpha=0.2, axis="y")

from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(facecolor=group_colors["Novice"], alpha=0.5, label="Novice"),
                        Patch(facecolor=group_colors["Expert"], alpha=0.5, label="Expert")],
               loc="best", fontsize=10)
fig.suptitle("TVA parameters from INDEPENDENT per-participant fits (no partial pooling)\n"
             "Brackets show uncorrected and Holm-adjusted p-values",
             fontsize=14, y=1.04)
fig.tight_layout()
savefig("FigS09_tva_parameters_independent")
plt.show()

## 10. Where each manuscript figure and table is produced

Figure 1 (typeface specimen) and Table 1 (rendered stimulus size) are not produced by this notebook.

| Manuscript item | Notebook section | Output file |
|-----------------|------------------|-------------|
| Figure 2 — posterior TVA curves with empirical means | 4.2 | `figures/Fig02_posterior_tva_curves.png` |
| Figure 3 — TVA parameters by Font × Group (hierarchical estimates) | 5.3 | `figures/Fig03_tva_parameters_hierarchical.png` |
| Figure 4 — shrinkage check | 9.3 | `figures/Fig04_shrinkage_check.png` |
| Figure 5 — Expert − Novice posterior difference distributions | 4.6 | `figures/Fig05_expert_novice_posterior_differences.png` |
| Figure 6 — forest plot of population-level posteriors | 4.4 | `figures/Fig06_forest_plot_population_posterior.png` |
| Figure 7 — font effect by group | 4.7 | `figures/Fig07_font_effect_by_group.png` |
| Figure 8 — matched exposure-time pairs | 7.2 | `figures/Fig08_matched_exposure_pairs.png` |
| Figure 9 — posterior difference curves Δ(t) | 4.3 | `figures/Fig09_posterior_difference_curves.png` |
| Figure 10 — DTW bootstrap, noise floor and temporal lead | 9.5 | `figures/Fig10_dtw_bootstrap_temporal_lead.png` |
| Figure 11 — age × expertise scatter plots | 6 | `figures/Fig11_age_expertise_scatter.png` |
| Figure 12 — capacity (K) vs. age by group | 8 (D5) | `figures/Fig12_capacity_vs_age.png` |
| Table 2 — posterior contrasts (Expert − Novice) | 4.5 | `results/tables/Table2_posterior_contrasts.csv` |
| Table 3 — Expert vs. Novice on independently fitted parameters | 9.7 | `results/tables/Table3_independent_fits.csv` |
| Table 4 — Tukey HSD pairwise comparisons | 5.5 | `results/tables/Table4_tukey_hsd.csv` |
| Table 5 — 3 oldest experts vs. 3 youngest novices | 8 (D3) | `results/tables/Table5_age_extremes.csv` |

**Statistics quoted in the text**

| Manuscript passage | Notebook section |
|--------------------|------------------|
| §2.1 demographics; sensitivity power analysis | 2.2; 9.6 |
| §2.2 stimulus statistics (letter frequencies, distinct strings) | 9.6 |
| §3.1 duplicate audit; trial accounting; per-participant retention | 2.3; 2.5; 9.1 |
| §3.1 exposure-timing discrepancy and residual timing scatter | 2.4; 9.1 |
| §3.1 LOO-CV comparison and Pareto-k | 3.3; 3.5 |
| §3.1 sensitivity refit with Expert #8 reinstated | 9.4 |
| §3.2 mixed ANOVA, linear mixed models, font penalties | 5.5 |
| §3.2 latent-scale font effects and interaction; natural-scale group means | 4.8; 4.9 |
| §3.2 ceiling rates, credible-interval widths and ROPE check | 9.2 |
| §3.2 shrinkage ratios and variance ratios | 9.3; 9.7 |
| §3.3 duration-matched comparisons (with Holm correction) | 7.2 |
| §3.3 posterior difference curves at 50, 100 and 200 ms | 4.3 |
| §3.3 DTW distances, bootstrap CI, noise floor, temporal lead | 8; 9.5 |
| §3.4 age differences, correlations and ANCOVA-style mixed models | 6; 8 (D1–D4); 5.5 |

**Supplementary figures** (not shown in the manuscript)

| File | Section | Content |
|------|---------|---------|
| `figures/FigS01_novice_timing_confusion_matrix.png` | 2.6 | Novice intended → corrected confusion matrix and retained trials |
| `figures/FigS02_posterior_difference_histograms.png` | 4.3 | Posterior Δ(t) histograms at selected exposure times |
| `figures/FigS03_expert_novice_differences_by_font.png` | 5.3 | Expert − Novice differences by font (hierarchical estimates) |
| `figures/FigS04_posterior_difference_violins.png` | 5.4 | Posterior difference violin plots with 95% intervals |
| `figures/FigS05_dtw_alignment_empirical.png` | 8 | DTW alignment of the empirical curves |
| `figures/FigS06_dtw_posterior_curves.png` | 8 | DTW on posterior TVA curves |
| `figures/FigS07_age_bands_tva_parameters.png` | 8 | TVA parameters across age bands by group |
| `figures/FigS08_group_by_intended_exposure.png` | 7.1 | Group performance at intended exposure times |
| `figures/FigS09_tva_parameters_independent.png` | 9.7 | TVA parameters from the independent per-participant fits |

**Design coding**

| Condition | isKlimax | isExpert | isKlimax:isExpert |
|---|---|---|---|
| Helvetica + Novice | 0 | 0 | 0 |
| Helvetica + Expert | 0 | 1 | 0 |
| Klimax Plus + Novice | 1 | 0 | 0 |
| Klimax Plus + Expert | 1 | 1 | 1 |
